# Phase 3.2 — Session 1: Byzantine-Robust Aggregation Comparison

Runs **7 aggregation methods** sequentially for 40 FL rounds each.
No diffusion. No blockchain for methods A–F. Ganache only for method G.

| # | Method | Defense Mechanism | ~Time |
|:-:|---|---|---:|
| A | FedAvg | None (defenseless baseline) | ~27 min |
| B | Krum (f=2) | Distance-based single selection | ~30 min |
| C | Multi-Krum (f=2, k=7) | Distance-based top-7 selection | ~30 min |
| D | Coordinate-wise Median | Per-parameter median | ~28 min |
| E | Trimmed Mean (β=0.2) | Trim outliers, average rest | ~28 min |
| F | Bulyan (f=1) | Iterative Krum → Trimmed Mean | ~33 min |
| G | Active-Ledger PoC-Only | On-chain trust scoring, Top-7 | ~35 min |
| | **Total** | | **~3.5 hours** |

> ⚠️ **Select GPU T4 ×2 and enable Internet before running.**

**Reference (Run C, already done):** Active-Ledger PoC + Diffusion → Mean F1 = **0.89**

In [1]:
# ============================================================
# Cell 1: GPU + Environment Verification
# ============================================================
import torch

print('=' * 60)
print('ENVIRONMENT CHECK')
print('=' * 60)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc   = torch.cuda.get_device_capability(0)
    print(f'GPU             : {name}')
    print(f'VRAM            : {vram:.1f} GB')
    print(f'Compute Cap     : {cc[0]}.{cc[1]}')
    if cc >= (7, 0):
        print('✅ GPU supported — everything runs on CUDA')
    else:
        print('⚠️  Older GPU — may fall back to CPU for some ops')
else:
    print('⚠️  No GPU — training will be on CPU (slower but works)')

print('=' * 60)

ENVIRONMENT CHECK
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB
Compute Cap     : 7.5
✅ GPU supported — everything runs on CUDA


In [3]:
# ============================================================
# Cell 2: Install Dependencies
# NOTE: wfdb, flwr, web3 are needed for data loading and
#       method G (PoC-Only uses Ganache + smart contract).
#       Diffusers are NOT needed for Session 1 (no diffusion).
# ============================================================
print('Installing dependencies...')

# Core FL and data dependencies
!pip install -q wfdb flwr web3 py-solc-x pyyaml scikit-learn

# Solidity compiler for smart contract deployment (method G)
from solcx import install_solc
install_solc('0.8.19')
print('✅ Solidity compiler 0.8.19 installed')

# Ganache (local Ethereum blockchain — needed only for method G)
!npm install -g ganache 2>/dev/null && echo '✅ Ganache installed' || echo '⚠️  Ganache install issue — method G may fail'

print('\n✅ All dependencies ready!')

Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.8/820.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 107.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 112.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━

In [3]:
# ============================================================
# Cell 3: Copy Repository Code to Working Directory
# Upload your repo as a Kaggle Dataset before running.
# The script handles nested zip extraction automatically.
# ============================================================
import shutil, os, glob

DATASET_PATHS = glob.glob('/kaggle/input/*')
print(f'Available datasets: {DATASET_PATHS}')

if not DATASET_PATHS:
    raise FileNotFoundError(
        'No dataset found!\n'
        'Click "Add Data" → upload your zipped FYP-Blockchain-FL folder as a dataset.'
    )

# Auto-detect repo root (handles Kaggle nesting)
repo_root = None
for base in DATASET_PATHS:
    # Check base directly
    if os.path.isfile(os.path.join(base, 'main.py')):
        repo_root = base
        break
    # Check one level deep
    for item in os.listdir(base):
        candidate = os.path.join(base, item)
        if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'main.py')):
            repo_root = candidate
            break
    if repo_root:
        break

if repo_root is None:
    # Deep search (handles deeper nesting)
    for base in DATASET_PATHS:
        for root, dirs, files in os.walk(base):
            if root.count(os.sep) - base.count(os.sep) > 4:
                break
            if 'main.py' in files and 'core' in dirs:
                repo_root = root
                break
        if repo_root:
            break

if repo_root is None:
    raise FileNotFoundError(
        f'Cannot locate main.py in your dataset.\n'
        f'Dataset contents: {[os.listdir(p) for p in DATASET_PATHS]}'
    )

print(f'Repo root found: {repo_root}')

# Copy all files to /kaggle/working
WORK = '/kaggle/working'
copied = []
for item in os.listdir(repo_root):
    if item.startswith('.'): continue
    src = os.path.join(repo_root, item)
    dst = os.path.join(WORK, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    copied.append(item)

os.chdir(WORK)

# Verify critical files
critical = ['main.py', 'config.yaml', 'core', 'benchmarks', 'contracts']
missing  = [f for f in critical if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Missing after copy: {missing}. Check your dataset.')

# Verify the new Phase 3.2 files are present
phase32_files = [
    'core/robust_aggregation.py',
    'benchmarks/run_robust_baselines.py',
]
missing_32 = [f for f in phase32_files if not os.path.exists(f)]
if missing_32:
    raise FileNotFoundError(
        f'Phase 3.2 files missing: {missing_32}\n'
        f'Make sure you uploaded the latest code (with robust_aggregation.py).'
    )

print(f'\n✅ Code copied. All critical files present.')
print(f'   Working dir: {os.getcwd()}')
print(f'   core/robust_aggregation.py : ✅')
print(f'   benchmarks/run_robust_baselines.py : ✅')

Available datasets: ['/kaggle/input/datasets']
Repo root found: /kaggle/input/datasets/muhammadibrahimfarid/deepcompute/FYP-Blockchain-FL-phase3-deep-compute

✅ Code copied. All critical files present.
   Working dir: /kaggle/working
   core/robust_aggregation.py : ✅
   benchmarks/run_robust_baselines.py : ✅


In [4]:
# ============================================================
# Cell 4: Download + Preprocess + Partition Data
# Downloads MIT-BIH Arrhythmia Database (~120 MB),
# segments into 360-sample ECG windows,
# and partitions across 10 clients (non-IID Dirichlet).
# ============================================================
print('=' * 60)
print('STEP 1/3: Downloading MIT-BIH Arrhythmia records...')
print('=' * 60)
!rm -rf data/* checkpoints/*
!PYTHONPATH=/kaggle/working python core/download_data.py

print('\n' + '=' * 60)
print('STEP 2/3: Preprocessing ECG signals...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/preprocess_data.py

print('\n' + '=' * 60)
print('STEP 3/3: Partitioning across 10 clients (non-IID)...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/partition_data.py

# Verify
import os
pdir = 'data/partitioned'
if os.path.exists(pdir):
    n_files = len(os.listdir(pdir))
    print(f'\n✅ Data ready — {n_files} client partition directories found')
else:
    print('❌ data/partitioned not found — check errors above')

STEP 1/3: Downloading MIT-BIH Arrhythmia records...

Target directory: /kaggle/working/data/mitdb

Downloading: 100%|██████████████████████████████| 48/48 [08:14<00:00, 10.30s/it]

Download complete! 48/48 records

Verifying downloaded files...
✅ Found 48 .dat files (ECG signals)
✅ Found 48 .hea files (headers)
✅ Found 48 .atr files (annotations)

🎉 All files downloaded successfully!

STEP 2/3: Preprocessing ECG signals...
Preprocessing ECG Data

Found 48 records
Window size: 360 samples

Processing records: 100%|███████████████████████| 48/48 [00:05<00:00,  8.63it/s]

Preprocessing Complete!

Class Distribution:
  Normal              : 75011 (74.83%)
  LBBB                :  8071 ( 8.05%)
  RBBB                :  7255 ( 7.24%)
  APB                 :  2781 ( 2.77%)
  PVC                 :  7129 ( 7.11%)

Total segments: 100247
Segment shape: (100247, 360)

✅ Saved to: data/processed/processed_data.pkl

STEP 3/3: Partitioning across 10 clients (non-IID)...

Loading preprocessed data fr

In [26]:
# ============================================================
# Cell 5: Start Ganache (needed ONLY for Method G: PoC-Only)
# Methods A–F do not use blockchain at all.
# The benchmark script starts Ganache mid-run when it reaches G.
# We start it here so it is ready and warm.
# ============================================================
import subprocess, time

print('Starting Ganache for Method G (Active-Ledger PoC-Only)...')
ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 '
    '--accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)

if ganache_proc.poll() is not None:
    print('Retrying with ganache-cli...')
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 '
        '--accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(10)

if ganache_proc.poll() is None:
    print(f'✅ Ganache running (PID: {ganache_proc.pid})')
else:
    print('⚠️  Ganache failed to start. Method G will run without blockchain.')
    print('   Methods A–F are unaffected.')

# Deploy FLLogger smart contract
print('\nDeploying FLLogger smart contract...')
!PYTHONPATH=/kaggle/working python core/deploy_contract.py
print('\n✅ Blockchain infrastructure ready!')

Starting Ganache for Method G (Active-Ledger PoC-Only)...
✅ Ganache running (PID: 57180)

Deploying FLLogger smart contract...

Deploying to Ganache

✅ Connected to Ganache
Block number: 0
Compiling Smart Contract

Compiling FLLogger.sol...
✅ Contract compiled successfully!

Deploying from account: 0x90F8bf6A479f320ead074411a4B0e7944Ea8c9C1

Deploying contract...
✅ Contract deployed at: 0xe78A0F7E598Cc8b0Bb87894B0F60dD2a88d6a8Ab
Gas used: 1,344,694

✅ Contract info saved to: contracts/deployed_contract.json

Testing Contract

Logging test update...
✅ Total updates: 1
✅ Test update retrieved:
   Round: 1
   Client: 1
   Accuracy: 95.80%

🎉 DEPLOYMENT COMPLETE!

Contract Address: 0xe78A0F7E598Cc8b0Bb87894B0F60dD2a88d6a8Ab
Save this address - you'll need it for FedAvg integration!

✅ Blockchain infrastructure ready!


In [13]:
# ============================================================
# Cell 5.5: THE FINAL BULLETPROOF HOTFIX PATCHER
# ============================================================
import os
import re
import shutil
import glob

print("Applying final bulletproof hotfixes...")

# 1. Find and Restore the pristine file to ensure a clean slate
repo_root = None
for base in glob.glob('/kaggle/input/*'):
    for root, dirs, files in os.walk(base):
        if 'main.py' in files:
            repo_root = root
            break
    if repo_root: break

if repo_root:
    pristine_file = os.path.join(repo_root, "benchmarks/run_robust_baselines.py")
    if os.path.exists(pristine_file):
        shutil.copy2(pristine_file, "benchmarks/run_robust_baselines.py")
        print("✅ Restored pristine benchmark file.")

# 2. Fix GPU Unlock in client.py
client_file = "core/client.py"
if os.path.exists(client_file):
    with open(client_file, "r") as f: code = f.read()
    code = code.replace('self.device = torch.device("cpu")', 'self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")')
    with open(client_file, "w") as f: f.write(code)
    print("✅ Hotfix 1 Applied: core/client.py now targets GPU.")

# 3. Apply Global Import and Loop Cleanup
runner_file = "benchmarks/run_robust_baselines.py"
if os.path.exists(runner_file):
    with open(runner_file, "r") as f: code = f.read()

    # Move imports to the top to avoid UnboundLocalError
    if "import gc" not in code:
        code = "import torch\nimport gc\n" + code
    
    # Inject only the cleanup commands inside the loop with dynamic indentation
    if "torch.cuda.empty_cache()" not in code:
        code = re.sub(
            r"^([ \t]*)(for \w+, \w+ in .*:)", 
            r"\1\2\n\1    gc.collect(); torch.cuda.empty_cache()", 
            code,
            flags=re.MULTILINE
        )
        with open(runner_file, "w") as f:
            f.write(code)
        print("✅ Hotfix 2 Applied: VRAM state bleeding patched.")

print("\n🚀 Ready for isolated execution!")

Applying final bulletproof hotfixes...
✅ Restored pristine benchmark file.
✅ Hotfix 1 Applied: core/client.py now targets GPU.
✅ Hotfix 2 Applied: VRAM state bleeding patched.

🚀 Ready for isolated execution!


---

## 🚀 Run Session 1 — All 7 Methods

The next cell runs everything. Expected output for each method:
```
======================================================================
METHOD: A_FedAvg
======================================================================
  --- Round   1/40 [A_FedAvg] ---
  Mean F1: 0.1711  |  Latency: 38.2s  |  Per-class: [0.855 0.000 0.000 0.000 0.000]
  ...
  [A_FedAvg] Finished. Total time: 0.45h
  [A_FedAvg] Final Mean F1: 0.1711

======================================================================
METHOD: B_Krum
...
```

Results are **saved after every method** — if Kaggle times out, the `.npy` file has all completed methods.

In [16]:
# ============================================================
# Cell 6: RUN SESSION 1 — All 7 Methods (Isolated & GPU Accelerated)
# ============================================================
import torch
import gc

# Initial manual wipe
gc.collect()
torch.cuda.empty_cache()

# Execute the robust baselines
!PYTHONPATH=/kaggle/working python main.py --mode robust-baselines

[MODE] Robust Baselines — delegating to benchmarks/run_robust_baselines.py
[LOG] All output is being saved to: checkpoints/session1_full.log
[LOG] Session started at: 2026-04-18 16:25:21
✅ GPU: Tesla T4
   VRAM: 15.6 GB

PHASE 3.2 — SESSION 1: ROBUST BASELINE COMPARISON
  Rounds            : 40
  Clients           : 8 honest + 2 Byzantine
  Methods           : 7 (A–G)
  Synthetic Data    : DISABLED for all methods in Session 1
  Device            : cuda

[..] Loading client data...
  Client  1:  20132 train samples
  Client  2:   8198 train samples
  Client  3:   1336 train samples
  Client  4:   5051 train samples
  Client  5:   3196 train samples
  Client  6:   2841 train samples
  Client  7:   8485 train samples
  Client  8:   4792 train samples
  Client  9:   2694 train samples
  Client 10:  13443 train samples
[OK] Loaded 10 clients | Total train samples: 70168
[OK] Global model initialized (seed=42). Will be reset for each method.

METHOD: A_FedAvg

  --- Round   1/40 [A_FedAvg] 

In [15]:
# ============================================================
# Cell 7: Display Results Summary & Integrity Audit
# ============================================================
import numpy as np
import os

results_path = 'checkpoints/robust_comparison.npy'
if not os.path.exists(results_path):
    print('❌ Results file not found. Check the console output of Cell 6.')
else:
    results = np.load(results_path, allow_pickle=True).item()
    print('=' * 75)
    print('SESSION 1 RESULTS — Final Per-Class F1 Scores (Round 40)')
    print('=' * 75)
    print(f"{'Method':<20} {'Normal':>7} {'LBBB':>7} {'RBBB':>7} {'APB':>7} {'PVC':>7} {'Mean F1':>9}")
    print('-' * 75)

    for name, res in results.items():
        f1      = res['final_f1']
        mean_f1 = float(np.mean(f1))
        
        # Integrity Check: Without diffusion, no method should exceed 0.70 Mean F1
        status = " ✅ HONEST" if mean_f1 < 0.70 else " ⚠️ LEAK DETECTED"
        if name == 'A_FedAvg': status = ""

        print(f"{name:<20} " + ' '.join(f'{v:7.3f}' for v in f1) + f" {mean_f1:9.4f}{status}")

    print('-' * 75)
    print(f"{'(Run C: PoC+Diffusion)':<20}   0.970   0.968   0.915   0.654   0.933     0.8887  [Phase 3.1]")
    print('=' * 75)

❌ Results file not found. Check the console output of Cell 6.


In [ ]:
# ============================================================
# Cell 8: Display Comparison Plot (inline)
# ============================================================
from IPython.display import Image, display
import os

if os.path.exists('session1_robust_comparison.png'):
    display(Image('session1_robust_comparison.png'))
else:
    print('Plot not generated yet. Run Cell 6 first.')

In [ ]:
# ============================================================
# Cell 9: Print LaTeX Table for Copy-Paste into Paper
# ============================================================
import os

tex_path = 'session1_latex_table.tex'
if os.path.exists(tex_path):
    with open(tex_path) as f:
        print(f.read())
else:
    print('LaTeX table not generated yet. Run Cell 6 first.')

In [17]:
%%writefile benchmarks/run_semantic_attacks.py
import os
import sys
import time
from copy import deepcopy
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report

sys.path.insert(0, str(Path(__file__).parent.parent))

from core.train_utils import load_client_data, create_data_loaders
from core.model import create_model
from core.utils import load_config
# Importing the updated client
from core.clientnew import FLClient 
from core.robust_aggregation import fedavg_aggregate, krum_aggregate, multi_krum_aggregate, median_aggregate, trimmed_mean_aggregate, bulyan_aggregate

NUM_ROUNDS = 40
NUM_NORMAL = 8
TOTAL_CLIENTS = 10
GANACHE_URL = "http://127.0.0.1:8545"

ATTACK_MODE = os.environ.get("ATTACK_MODE", "label_flip")
SLEEPER_ACTIVATION_ROUND = int(os.environ.get("SLEEPER_ACTIVATION_ROUND", "15"))

METHODS = OrderedDict([
    ("A_FedAvg", (fedavg_aggregate, {})),
    ("B_Krum", (krum_aggregate, {"f": 2})),
    ("C_MultiKrum", (multi_krum_aggregate, {"f": 2, "k": 7})),
    ("D_Median", (median_aggregate, {})),
    ("E_TrimmedMean", (trimmed_mean_aggregate, {"beta": 0.2})),
    ("F_Bulyan", (bulyan_aggregate, {"f": 1})),
])
POC_METHOD_KEY = "G_PoC_Only"

def _get_weights(model): return [val.cpu().detach().numpy() for _, val in model.state_dict().items()]
def _set_weights(model, weights):
    state_dict = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(state_dict, strict=True)

def evaluate_full(global_model, val_loaders, device):
    """Restored full telemetry evaluation."""
    global_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for loader in val_loaders:
            for X, y in loader:
                X, y = X.to(device), y.to(device)
                preds = torch.argmax(global_model(X), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                
    f1 = f1_score(all_labels, all_preds, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
    precision, recall, _, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=[0, 1, 2, 3, 4], zero_division=0
    )
    return f1, precision, recall, support

def run_one_method(method_name, agg_fn, agg_kwargs, global_model_init, loaders, val_loaders, sizes, device, config, malicious_ids, attack_mode, sleeper_activation_round, blockchain=None):
    print(f"\n{'='*70}\nMETHOD: {method_name} | ATTACK: {attack_mode}\n{'='*70}")
    global_model = deepcopy(global_model_init)
    global_model.to(device)
    round_f1s = []

    for rnd in range(1, NUM_ROUNDS + 1):
        t_rnd = time.time()
        print(f"\n  --- Round {rnd:3d}/{NUM_ROUNDS} [{method_name}] ---")
        
        global_weights = _get_weights(global_model)
        results = []
        for cid in range(TOTAL_CLIENTS):
            client = FLClient(
                client_id=cid + 1, model=global_model, train_loader=loaders[cid], val_loader=val_loaders[cid],
                config=config, is_malicious=(cid in malicious_ids), blockchain_manager=blockchain,
                attack_mode=attack_mode, sleeper_activation_round=sleeper_activation_round, current_round=rnd
            )
            w, n, metrics = client.fit(global_weights, config)
            results.append((float(metrics["accuracy"]), w, n, cid))

        if blockchain is not None:
            from core.blockchain import fetch_client_history
            from core.server import calculate_score
            eth_accounts = list(blockchain.w3.eth.accounts)
            while len(eth_accounts) < TOTAL_CLIENTS: eth_accounts.append(blockchain.deployer)
            for acc, w, n, cid in results:
                try:
                    tx = blockchain.contract.functions.logUpdate(rnd, cid + 1, bytes([cid % 256] * 32), n, int(acc * 10000)).transact({"from": eth_accounts[cid]})
                    blockchain.w3.eth.wait_for_transaction_receipt(tx)
                except Exception: pass
            scored = []
            for acc, w, n, cid in results:
                try: score = calculate_score(fetch_client_history(eth_accounts[cid], blockchain.contract, blockchain.w3))
                except Exception: score = 0.5
                scored.append((score, w, n))
            scored.sort(key=lambda x: x[0], reverse=True)
            top7 = scored[:7]
            global_model = fedavg_aggregate(global_model, [w for _, w, _ in top7], [n for _, _, n in top7])
        else:
            global_model = agg_fn(global_model, [w for _, w, _, _ in results], [n for _, _, n, _ in results], **agg_kwargs)

        f1_now, prec_now, rec_now, sup_now = evaluate_full(global_model, val_loaders, device)
        mean_f1 = float(np.mean(f1_now))
        t_elapsed = time.time() - t_rnd
        round_f1s.append(mean_f1)
        
        # Restored detailed printing
        print(f"  Mean F1: {mean_f1:.4f}  |  Latency: {t_elapsed:.1f}s")
        print(f"  Per-class F1       : {np.round(f1_now, 4)}")
        print(f"  Per-class Precision: {np.round(prec_now, 4)}")
        print(f"  Per-class Recall   : {np.round(rec_now, 4)}")
        print(f"  Support            : {sup_now}")

    final_f1, _, _, _ = evaluate_full(global_model, val_loaders, device)
    print(f"\n  [{method_name}] Final Mean F1: {float(np.mean(final_f1)):.4f}")
    return {"round_f1": np.array(round_f1s), "final_f1": final_f1}

def main():
    torch.manual_seed(42); np.random.seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = load_config()
    loaders, val_loaders, sizes = [], [], []
    for cid in range(1, TOTAL_CLIENTS + 1):
        data = load_client_data(cid, config["data"]["partitioned_dir"])
        tl, vl = create_data_loaders(data["X_train"], data["y_train"], data["X_val"], data["y_val"], config["training"]["batch_size"])
        loaders.append(tl); val_loaders.append(vl); sizes.append(len(data["y_train"]))

    global_model_init = create_model(config)
    init_weights = _get_weights(global_model_init)
    all_results = {}
    
    for method_key, (agg_fn, agg_kwargs) in METHODS.items():
        fresh_model = create_model(config)
        _set_weights(fresh_model, init_weights)
        all_results[method_key] = run_one_method(method_key, agg_fn, agg_kwargs, fresh_model, loaders, val_loaders, sizes, device, config, {8, 9}, ATTACK_MODE, SLEEPER_ACTIVATION_ROUND)
        import gc; gc.collect(); torch.cuda.empty_cache()

    try:
        from core.blockchain import BlockchainManager
        blockchain = BlockchainManager(GANACHE_URL)
        fresh_model_g = create_model(config)
        _set_weights(fresh_model_g, init_weights)
        all_results[POC_METHOD_KEY] = run_one_method(POC_METHOD_KEY, fedavg_aggregate, {}, fresh_model_g, loaders, val_loaders, sizes, device, config, {8, 9}, ATTACK_MODE, SLEEPER_ACTIVATION_ROUND, blockchain)
    except Exception:
        print("[WARN] Ganache failed. Skipping PoC method.")

if __name__ == "__main__":
    main()

Overwriting benchmarks/run_semantic_attacks.py


In [16]:
%%writefile core/clientnew.py
"""
Flower Client for FL — Tri-Mode Byzantine Attack Simulation with Perfect Collusion.
"""

import os
import sys
import numpy as np
import torch
import torch.nn as nn
from copy import deepcopy
from pathlib import Path
from typing import Dict, Tuple
from torch.utils.data import TensorDataset, DataLoader

sys.path.insert(0, str(Path(__file__).parent.parent))

import flwr as fl
from flwr.common import NDArrays, Scalar
from core.model import CNNLSTM
from core.train_utils import train_epoch, evaluate

class FLClient(fl.client.NumPyClient):
    def __init__(
        self, client_id: int, model: CNNLSTM, train_loader, val_loader, config: Dict,
        is_malicious: bool = False, blockchain_manager=None,
        enable_synthetic: bool = False, diffusion_steps: int = 2, synthetic_quantity: int = 2,
        attack_mode: str = 'label_flip', sleeper_activation_round: int = 15, current_round: int = 1
    ):
        self.client_id = client_id
        self.model = deepcopy(model)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.is_malicious = is_malicious
        self.blockchain = blockchain_manager
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        self.enable_synthetic = enable_synthetic
        self.attack_mode = attack_mode
        self.sleeper_activation_round = sleeper_activation_round
        self.current_round = current_round

        # 🚨 THE KRUM KILLER: Synchronizes dataset sizes so attackers cluster perfectly
        self.collusion_sync_limit = 3000

    def get_parameters(self, config: Dict) -> NDArrays:
        return [val.cpu().detach().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters: NDArrays) -> None:
        state_dict = dict(zip(self.model.state_dict().keys(), [torch.tensor(p) for p in parameters]))
        self.model.load_state_dict(state_dict, strict=True)

    def _compute_class_weights(self, loader):
        """Dynamically calculates inverse-frequency weights for the current loader."""
        class_counts = {i: 0 for i in range(5)}
        for _, label in loader.dataset:
            label_val = int(label.item() if hasattr(label, 'item') else label)
            if label_val in class_counts:
                class_counts[label_val] += 1
                
        total = sum(class_counts.values())
        if total == 0: return None
        
        weights = []
        for c in range(5):
            if class_counts[c] > 0:
                w = total / (5 * class_counts[c])
                weights.append(max(0.3, min(w, 10.0))) # Clamped between 0.3 and 10.0
            else:
                weights.append(1.0)
        return torch.FloatTensor(weights).to(self.device)

    def _apply_semantic_poison(self, loader, enforce_sync=False):
        """Flips Normal (0) <-> LBBB (1) and optionally syncs dataset size."""
        all_X = torch.cat([batch[0] for batch in loader])
        all_y = torch.cat([batch[1] for batch in loader])
        
        poisoned_y = all_y.clone()
        mask_0, mask_1 = (all_y == 0), (all_y == 1)
        poisoned_y[mask_0] = 1
        poisoned_y[mask_1] = 0
        
        # --- COLLUSION SYNC: Truncate so attackers end up perfectly clustered ---
        if enforce_sync:
            limit = min(self.collusion_sync_limit, len(all_X))
            all_X, poisoned_y = all_X[:limit], poisoned_y[:limit]
            
        os_workers = 0 if os.name == 'nt' else 2
        bs = self.config['training']['batch_size']
        
        return DataLoader(
            TensorDataset(all_X, poisoned_y), 
            batch_size=bs, shuffle=True, pin_memory=True, num_workers=os_workers
        )

    def _is_active_attacker(self) -> bool:
        """Determines if this client should act maliciously in the current round."""
        if not self.is_malicious:
            return False
        if self.attack_mode in ['gaussian', 'label_flip']:
            return True
        if self.attack_mode == 'sleeper' and self.current_round > self.sleeper_activation_round:
            return True
        return False

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict[str, Scalar]]:
        self.set_parameters(parameters)
        
        active_attacker = self._is_active_attacker()

        # Logging Sleeper Awakening
        if self.is_malicious and self.attack_mode == 'sleeper' and self.current_round == self.sleeper_activation_round + 1:
            print(f"  [ATTACK] Client {self.client_id} AWAKENED! Activating label-flip.")

        # Apply Poisoning
        if active_attacker and self.attack_mode in ['label_flip', 'sleeper']:
            if self.current_round == 1 or (self.attack_mode == 'sleeper' and self.current_round == self.sleeper_activation_round + 1):
                print(f"  [ATTACK] Client {self.client_id} applying Target Label-Flip (0<->1) with Collusion Sync.")
            self.train_loader = self._apply_semantic_poison(self.train_loader, enforce_sync=True)

        # Training Phase
        local_epochs = int(config.get("local_epochs", self.config["federated"]["local_epochs"]))
        lr = self.config["model"]["learning_rate"]
        criterion = nn.CrossEntropyLoss(weight=self._compute_class_weights(self.train_loader))
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)

        self.model.train()
        for _ in range(local_epochs):
            train_epoch(self.model, self.train_loader, criterion, optimizer, self.device)

        updated_weights = self.get_parameters({})
        num_samples = len(self.train_loader.dataset)

        # Post-Training Attack Behavior
        if self.is_malicious and self.attack_mode == 'gaussian':
            poisoned = [np.random.normal(0, 1, w.shape).astype(np.float32) for w in updated_weights]
            return poisoned, num_samples, {"accuracy": 0.10, "is_malicious": 1}

        elif active_attacker and self.attack_mode in ['label_flip', 'sleeper']:
            # Attackers evaluate on flipped val data to report fake high accuracy to the blockchain
            flipped_val = self._apply_semantic_poison(self.val_loader, enforce_sync=False)
            reported_acc = float(evaluate(self.model, flipped_val, nn.CrossEntropyLoss(), self.device)["accuracy"])
            return updated_weights, num_samples, {"accuracy": reported_acc, "is_malicious": 1}
        
        # Honest (or Dormant Sleeper) Behavior
        val_metrics = evaluate(self.model, self.val_loader, nn.CrossEntropyLoss(), self.device)
        return updated_weights, num_samples, {"accuracy": float(val_metrics["accuracy"]), "is_malicious": 0}

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict[str, Scalar]]:
        self.set_parameters(parameters)
        metrics = evaluate(self.model, self.val_loader, nn.CrossEntropyLoss(), self.device)
        return (float(metrics["loss"]), len(self.val_loader.dataset), {"accuracy": float(metrics["accuracy"]), "f1": float(metrics["f1"])})

Overwriting core/clientnew.py


In [18]:
# ============================================================
# Cell 12: Execute Session 2 (Label-Flip Attack)
# ============================================================
!ATTACK_MODE='label_flip' PYTHONPATH=/kaggle/working python benchmarks/run_semantic_attacks.py


METHOD: A_FedAvg | ATTACK: label_flip

  --- Round   1/40 [A_FedAvg] ---
  [ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.
  [ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.
  Mean F1: 0.0943  |  Latency: 22.5s
  Per-class F1       : [0.0699 0.1324 0.     0.     0.2695]
  Per-class Precision: [0.8249 0.0753 0.     0.     0.1604]
  Per-class Recall   : [0.0365 0.5488 0.     0.     0.8421]
  Support            : [11237  1210  1085   416  1089]

  --- Round   2/40 [A_FedAvg] ---
  Mean F1: 0.5087  |  Latency: 21.4s
  Per-class F1       : [0.8692 0.5354 0.3428 0.     0.7963]
  Per-class Precision: [0.975  0.3926 0.2689 0.     0.6885]
  Per-class Recall   : [0.7842 0.8413 0.4728 0.     0.944 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round   3/40 [A_FedAvg] ---
  Mean F1: 0.5306  |  Latency: 20.8s
  Per-class F1       : [0.7916 0.6627 0.3254 0.0183 0.8547]
  Per-class Precision: [0.9841 0.5149 0.2071 0.1905 0.7873]
  

In [38]:
import numpy as np
import pandas as pd
import re
import os


# --- PASTE YOUR TERMINAL OUTPUT HERE ---
terminal_output = """
======================================================================

METHOD: A_FedAvg | ATTACK: label_flip

======================================================================



--- Round 1/40 [A_FedAvg] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

Mean F1: 0.0943 | Latency: 22.5s

Per-class F1 : [0.0699 0.1324 0. 0. 0.2695]

Per-class Precision: [0.8249 0.0753 0. 0. 0.1604]

Per-class Recall : [0.0365 0.5488 0. 0. 0.8421]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [A_FedAvg] ---

Mean F1: 0.5087 | Latency: 21.4s

Per-class F1 : [0.8692 0.5354 0.3428 0. 0.7963]

Per-class Precision: [0.975 0.3926 0.2689 0. 0.6885]

Per-class Recall : [0.7842 0.8413 0.4728 0. 0.944 ]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [A_FedAvg] ---

Mean F1: 0.5306 | Latency: 20.8s

Per-class F1 : [0.7916 0.6627 0.3254 0.0183 0.8547]

Per-class Precision: [0.9841 0.5149 0.2071 0.1905 0.7873]

Per-class Recall : [0.662 0.9298 0.7594 0.0096 0.9348]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [A_FedAvg] ---

Mean F1: 0.6139 | Latency: 20.6s

Per-class F1 : [0.8889 0.7391 0.5202 0.0585 0.8628]

Per-class Precision: [0.9835 0.608 0.3725 0.2222 0.7863]

Per-class Recall : [0.8109 0.9421 0.8618 0.0337 0.9559]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [A_FedAvg] ---

Mean F1: 0.7476 | Latency: 20.6s

Per-class F1 : [0.9641 0.9179 0.916 0.1129 0.8271]

Per-class Precision: [0.9745 0.8784 0.8842 0.35 0.7205]

Per-class Recall : [0.9539 0.9612 0.9502 0.0673 0.9706]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [A_FedAvg] ---

Mean F1: 0.7998 | Latency: 20.7s

Per-class F1 : [0.9734 0.9154 0.8975 0.337 0.8757]

Per-class Precision: [0.9825 0.8631 0.8617 0.6838 0.7977]

Per-class Recall : [0.9645 0.9744 0.9364 0.2236 0.9706]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [A_FedAvg] ---

Mean F1: 0.8090 | Latency: 20.7s

Per-class F1 : [0.968 0.8912 0.9026 0.4354 0.8479]

Per-class Precision: [0.9889 0.8159 0.8409 0.6821 0.7642]

Per-class Recall : [0.9479 0.9818 0.9742 0.3197 0.9522]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [A_FedAvg] ---

Mean F1: 0.7568 | Latency: 20.7s

Per-class F1 : [0.9541 0.8678 0.821 0.2918 0.8492]

Per-class Precision: [0.9893 0.7807 0.706 0.5616 0.7537]

Per-class Recall : [0.9213 0.9769 0.9806 0.1971 0.9725]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [A_FedAvg] ---

Mean F1: 0.8983 | Latency: 20.8s

Per-class F1 : [0.9771 0.956 0.9712 0.7066 0.8805]

Per-class Precision: [0.989 0.9352 0.9629 0.7721 0.7964]

Per-class Recall : [0.9656 0.9777 0.9797 0.6514 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [A_FedAvg] ---

Mean F1: 0.9080 | Latency: 20.6s

Per-class F1 : [0.9811 0.951 0.966 0.737 0.9048]

Per-class Precision: [0.9899 0.9243 0.9517 0.8288 0.8424]

Per-class Recall : [0.9726 0.9793 0.9806 0.6635 0.977 ]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [A_FedAvg] ---

Mean F1: 0.8966 | Latency: 20.6s

Per-class F1 : [0.9764 0.9467 0.9572 0.7202 0.8823]

Per-class Precision: [0.9917 0.9184 0.9365 0.7533 0.7994]

Per-class Recall : [0.9615 0.9769 0.9788 0.6899 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [A_FedAvg] ---

Mean F1: 0.9014 | Latency: 20.7s

Per-class F1 : [0.9788 0.9353 0.9659 0.7261 0.9009]

Per-class Precision: [0.9918 0.8944 0.9533 0.7849 0.8299]

Per-class Recall : [0.9661 0.9802 0.9788 0.6755 0.9853]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [A_FedAvg] ---

Mean F1: 0.8952 | Latency: 20.7s

Per-class F1 : [0.9727 0.915 0.9463 0.7503 0.8916]

Per-class Precision: [0.9918 0.8555 0.9127 0.8123 0.8167]

Per-class Recall : [0.9543 0.9835 0.9825 0.6971 0.9816]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [A_FedAvg] ---

Mean F1: 0.9115 | Latency: 20.7s

Per-class F1 : [0.9802 0.9688 0.9692 0.7464 0.8929]

Per-class Precision: [0.9928 0.9625 0.952 0.75 0.815 ]

Per-class Recall : [0.9679 0.9752 0.9871 0.7428 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [A_FedAvg] ---

Mean F1: 0.9198 | Latency: 20.7s

Per-class F1 : [0.9823 0.9558 0.9783 0.7712 0.9114]

Per-class Precision: [0.9915 0.9297 0.9797 0.8133 0.8465]

Per-class Recall : [0.9732 0.9835 0.977 0.7332 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [A_FedAvg] ---

Mean F1: 0.9317 | Latency: 20.6s

Per-class F1 : [0.9854 0.9671 0.979 0.7959 0.9311]

Per-class Precision: [0.9921 0.9512 0.9701 0.8418 0.8811]

Per-class Recall : [0.9789 0.9835 0.988 0.7548 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [A_FedAvg] ---

Mean F1: 0.9387 | Latency: 20.6s

Per-class F1 : [0.9867 0.9694 0.9839 0.8175 0.9362]

Per-class Precision: [0.9921 0.9588 0.9843 0.8516 0.8873]

Per-class Recall : [0.9814 0.9802 0.9834 0.7861 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [A_FedAvg] ---

Mean F1: 0.9404 | Latency: 20.6s

Per-class F1 : [0.9876 0.9782 0.9826 0.8064 0.9472]

Per-class Precision: [0.9922 0.9738 0.9763 0.8225 0.9096]

Per-class Recall : [0.9831 0.9826 0.9889 0.7909 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [A_FedAvg] ---

Mean F1: 0.9325 | Latency: 20.6s

Per-class F1 : [0.9843 0.9721 0.9817 0.8024 0.9219]

Per-class Precision: [0.9925 0.9657 0.9772 0.8093 0.8612]

Per-class Recall : [0.9762 0.9785 0.9862 0.7957 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [A_FedAvg] ---

Mean F1: 0.9369 | Latency: 20.5s

Per-class F1 : [0.9868 0.977 0.9813 0.7966 0.9429]

Per-class Precision: [0.9924 0.9691 0.9728 0.8074 0.9033]

Per-class Recall : [0.9811 0.9851 0.9899 0.7861 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [A_FedAvg] ---

Mean F1: 0.9373 | Latency: 20.6s

Per-class F1 : [0.9869 0.9671 0.9829 0.804 0.9456]

Per-class Precision: [0.9921 0.9505 0.9834 0.8308 0.9058]

Per-class Recall : [0.9818 0.9843 0.9825 0.7788 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [A_FedAvg] ---

Mean F1: 0.9391 | Latency: 20.5s

Per-class F1 : [0.9878 0.9746 0.9848 0.8015 0.9466]

Per-class Precision: [0.9921 0.9652 0.9835 0.8395 0.9046]

Per-class Recall : [0.9835 0.9843 0.9862 0.7668 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [A_FedAvg] ---

Mean F1: 0.9371 | Latency: 20.6s

Per-class F1 : [0.987 0.9678 0.9835 0.8039 0.9431]

Per-class Precision: [0.9931 0.9542 0.9781 0.82 0.9013]

Per-class Recall : [0.981 0.9818 0.9889 0.7885 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [A_FedAvg] ---

Mean F1: 0.9365 | Latency: 20.5s

Per-class F1 : [0.9855 0.9584 0.9794 0.8063 0.9527]

Per-class Precision: [0.9936 0.9287 0.9719 0.8072 0.9182]

Per-class Recall : [0.9776 0.9901 0.9871 0.8053 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [A_FedAvg] ---

Mean F1: 0.9356 | Latency: 20.6s

Per-class F1 : [0.9863 0.971 0.9777 0.8077 0.935 ]

Per-class Precision: [0.9925 0.9589 0.9642 0.8654 0.8852]

Per-class Recall : [0.9802 0.9835 0.9917 0.7572 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [A_FedAvg] ---

Mean F1: 0.9465 | Latency: 20.7s

Per-class F1 : [0.9897 0.9751 0.9839 0.8225 0.9612]

Per-class Precision: [0.9935 0.9644 0.9799 0.8379 0.9341]

Per-class Recall : [0.9859 0.986 0.988 0.8077 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [A_FedAvg] ---

Mean F1: 0.9349 | Latency: 20.5s

Per-class F1 : [0.9849 0.976 0.9844 0.819 0.9102]

Per-class Precision: [0.9917 0.9768 0.9782 0.8788 0.8411]

Per-class Recall : [0.9781 0.9752 0.9908 0.7668 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [A_FedAvg] ---

Mean F1: 0.9444 | Latency: 20.6s

Per-class F1 : [0.9891 0.9806 0.9836 0.8077 0.961 ]

Per-class Precision: [0.9926 0.9778 0.9729 0.8127 0.9379]

Per-class Recall : [0.9855 0.9835 0.9945 0.8029 0.9853]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [A_FedAvg] ---

Mean F1: 0.9355 | Latency: 20.6s

Per-class F1 : [0.9844 0.9692 0.9822 0.8048 0.9368]

Per-class Precision: [0.9926 0.9522 0.9737 0.8019 0.8914]

Per-class Recall : [0.9764 0.9868 0.9908 0.8077 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [A_FedAvg] ---

Mean F1: 0.9537 | Latency: 20.5s

Per-class F1 : [0.9906 0.984 0.9881 0.8349 0.9707]

Per-class Precision: [0.9936 0.9795 0.9827 0.831 0.9539]

Per-class Recall : [0.9877 0.9884 0.9935 0.8389 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [A_FedAvg] ---

Mean F1: 0.9530 | Latency: 20.5s

Per-class F1 : [0.9909 0.9827 0.9845 0.839 0.968 ]

Per-class Precision: [0.9938 0.9771 0.9747 0.8515 0.9496]

Per-class Recall : [0.9879 0.9884 0.9945 0.8269 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [A_FedAvg] ---

Mean F1: 0.9456 | Latency: 20.6s

Per-class F1 : [0.9876 0.9649 0.984 0.8321 0.9594]

Per-class Precision: [0.9933 0.9425 0.9764 0.8424 0.9332]

Per-class Recall : [0.9819 0.9884 0.9917 0.8221 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [A_FedAvg] ---

Mean F1: 0.9532 | Latency: 20.6s

Per-class F1 : [0.9914 0.9864 0.9849 0.8292 0.9741]

Per-class Precision: [0.9931 0.986 0.9799 0.8242 0.9649]

Per-class Recall : [0.9898 0.9868 0.9899 0.8341 0.9835]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [A_FedAvg] ---

Mean F1: 0.9572 | Latency: 20.6s

Per-class F1 : [0.9918 0.9872 0.9863 0.8472 0.9737]

Per-class Precision: [0.9925 0.9884 0.9791 0.8766 0.9607]

Per-class Recall : [0.9911 0.986 0.9935 0.8197 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [A_FedAvg] ---

Mean F1: 0.9543 | Latency: 20.7s

Per-class F1 : [0.9916 0.9855 0.9862 0.8337 0.9746]

Per-class Precision: [0.9925 0.9851 0.9826 0.8483 0.9632]

Per-class Recall : [0.9907 0.986 0.9899 0.8197 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [A_FedAvg] ---

Mean F1: 0.9491 | Latency: 20.7s

Per-class F1 : [0.9897 0.9815 0.9849 0.8264 0.9629]

Per-class Precision: [0.9929 0.9771 0.9782 0.8408 0.9374]

Per-class Recall : [0.9865 0.986 0.9917 0.8125 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [A_FedAvg] ---

Mean F1: 0.9588 | Latency: 20.5s

Per-class F1 : [0.9924 0.9831 0.9859 0.8546 0.9782]

Per-class Precision: [0.9923 0.9827 0.9765 0.9103 0.9677]

Per-class Recall : [0.9925 0.9835 0.9954 0.8053 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [A_FedAvg] ---

Mean F1: 0.9525 | Latency: 20.8s

Per-class F1 : [0.9906 0.9864 0.9863 0.8241 0.9751]

Per-class Precision: [0.9942 0.9868 0.98 0.7946 0.96 ]

Per-class Recall : [0.9871 0.986 0.9926 0.8558 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [A_FedAvg] ---

Mean F1: 0.9466 | Latency: 20.7s

Per-class F1 : [0.9895 0.9795 0.9796 0.8231 0.9613]

Per-class Precision: [0.9943 0.9708 0.9651 0.8417 0.9319]

Per-class Recall : [0.9848 0.9884 0.9945 0.8053 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [A_FedAvg] ---

Mean F1: 0.9613 | Latency: 20.7s

Per-class F1 : [0.9927 0.9892 0.9899 0.8551 0.9794]

Per-class Precision: [0.9935 0.9909 0.9863 0.852 0.9745]

Per-class Recall : [0.9919 0.9876 0.9935 0.8582 0.9844]

Support : [11237 1210 1085 416 1089]



[A_FedAvg] Final Mean F1: 0.9613



======================================================================

METHOD: B_Krum | ATTACK: label_flip

======================================================================



--- Round 1/40 [B_Krum] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

[Krum] Winner: client 5 | Scores: [2188504.65 174523.08 60087.41 47674.25 17112.57 18684.6

194166.43 43665.87 20092.27 17702.86]

Mean F1: 0.3070 | Latency: 20.8s

Per-class F1 : [0.7494 0.2783 0.0769 0. 0.4303]

Per-class Precision: [0.8933 0.1922 0.2945 0. 0.2806]

Per-class Recall : [0.6455 0.5041 0.0442 0. 0.9229]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185853.41 173626.39 59768.28 47020.52 16685.59 18305.1

193001.45 43114.34 19901.1 17456.12]

Mean F1: 0.5349 | Latency: 20.7s

Per-class F1 : [0.6351 0.7349 0.2922 0.1719 0.8405]

Per-class Precision: [0.9557 0.6766 0.1726 0.1247 0.839 ]

Per-class Recall : [0.4756 0.8041 0.953 0.2764 0.8421]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185841.58 173459.62 59727.62 46922.36 16615.6 18241.5

192899.93 43012.46 19774.13 17456.05]

Mean F1: 0.6116 | Latency: 20.7s

Per-class F1 : [0.788 0.748 0.5303 0.1957 0.796 ]

Per-class Precision: [0.9723 0.6943 0.3708 0.1216 0.6806]

Per-class Recall : [0.6625 0.8107 0.9309 0.5 0.9587]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185534.07 173368.92 59768.56 46944.91 16657.25 18251.23

192881.23 43031.1 19824.58 17542.45]

Mean F1: 0.6904 | Latency: 20.6s

Per-class F1 : [0.8635 0.7718 0.6311 0.3271 0.8586]

Per-class Precision: [0.9847 0.6687 0.4642 0.234 0.7774]

Per-class Recall : [0.7689 0.9124 0.9853 0.5433 0.9587]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185441.99 173379.01 59719.91 46923.83 16640.04 18241.85

192967.07 43031.1 19780.11 17436.32]

Mean F1: 0.7702 | Latency: 20.6s

Per-class F1 : [0.9383 0.8637 0.7927 0.3717 0.8843]

Per-class Precision: [0.9834 0.8065 0.6623 0.3393 0.8213]

Per-class Recall : [0.8971 0.9298 0.9871 0.4111 0.9578]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185344.68 173381.62 59748.23 46966.24 16640.74 18269.82

192803.93 43081.21 19851.62 17446.81]

Mean F1: 0.7978 | Latency: 20.8s

Per-class F1 : [0.943 0.7974 0.9411 0.4733 0.8344]

Per-class Precision: [0.9833 0.6863 0.926 0.4492 0.7365]

Per-class Recall : [0.9059 0.9512 0.9567 0.5 0.9624]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185375.12 173324.76 59762.45 46898.94 16632.78 18266.83

192706.85 43042.34 19859.28 17365.17]

Mean F1: 0.8266 | Latency: 20.6s

Per-class F1 : [0.9645 0.8654 0.9563 0.4462 0.9006]

Per-class Precision: [0.9794 0.7865 0.9349 0.4218 0.9156]

Per-class Recall : [0.9501 0.962 0.9788 0.4736 0.8861]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185315.72 173380.8 59817.15 46966.74 16702.26 18292.63

192824.87 43042.55 19843.04 17436.86]

Mean F1: 0.7767 | Latency: 20.6s

Per-class F1 : [0.928 0.7901 0.8319 0.4814 0.8523]

Per-class Precision: [0.9864 0.6777 0.7164 0.4307 0.7738]

Per-class Recall : [0.8762 0.9471 0.9917 0.5457 0.9486]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185188.13 173410.19 59811.12 46971.03 16663.96 18296.88

192871.35 43060.87 19815.37 17438.06]

Mean F1: 0.8782 | Latency: 20.7s

Per-class F1 : [0.9739 0.9033 0.949 0.6578 0.9069]

Per-class Precision: [0.9766 0.9232 0.9145 0.7373 0.8656]

Per-class Recall : [0.9712 0.8843 0.9862 0.5938 0.9522]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185836.49 173406.32 59845.37 46980.48 16735.29 18322.14

192811.32 43096.21 19862.59 17493.08]

Mean F1: 0.8791 | Latency: 20.7s

Per-class F1 : [0.9774 0.9383 0.9692 0.6019 0.9087]

Per-class Precision: [0.983 0.9531 0.952 0.6005 0.8629]

Per-class Recall : [0.9719 0.924 0.9871 0.6034 0.9596]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186059.53 173446.09 59818.8 46997.6 16695.76 18311.28

192828.49 43114.14 19829.61 17474.14]

Mean F1: 0.8585 | Latency: 20.6s

Per-class F1 : [0.9658 0.9101 0.9507 0.637 0.8289]

Per-class Precision: [0.9874 0.8738 0.9503 0.6548 0.7177]

Per-class Recall : [0.945 0.9496 0.9512 0.6202 0.9807]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185802.02 173398.96 59844.89 46971.83 16787.85 18329.43

192790.39 43093.78 19879.46 17502.11]

Mean F1: 0.8887 | Latency: 20.6s

Per-class F1 : [0.9806 0.935 0.9592 0.6619 0.9067]

Per-class Precision: [0.9838 0.9311 0.9329 0.8244 0.8563]

Per-class Recall : [0.9775 0.9388 0.9871 0.5529 0.9633]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185604.2 173443.78 60059.21 47047.99 16830.19 18408.23

192988.8 43189.12 19985.99 17688.63]

Mean F1: 0.8892 | Latency: 20.7s

Per-class F1 : [0.9804 0.9466 0.9301 0.663 0.9259]

Per-class Precision: [0.9847 0.9478 0.8743 0.7836 0.901 ]

Per-class Recall : [0.9762 0.9455 0.9935 0.5745 0.9522]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185396.52 173539.62 59959.57 47063.24 16805.49 18373.78

192890.35 43132.26 19967.95 17544.47]

Mean F1: 0.8799 | Latency: 20.6s

Per-class F1 : [0.9765 0.9634 0.9484 0.5839 0.9272]

Per-class Precision: [0.9844 0.9803 0.9088 0.5339 0.9128]

Per-class Recall : [0.9687 0.9471 0.9917 0.6442 0.9421]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185249.11 173597.91 59984.21 47143.23 16849. 18440.02

192989.39 43206.62 20079.78 17670.62]

Mean F1: 0.8674 | Latency: 20.8s

Per-class F1 : [0.9722 0.9335 0.938 0.6002 0.8929]

Per-class Precision: [0.9863 0.9706 0.8913 0.5513 0.8228]

Per-class Recall : [0.9586 0.8992 0.9899 0.6587 0.9761]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185395.78 173610.15 59954.1 47189.98 16893.81 18466.85

193013.29 43231.79 20244.95 17773.23]

Mean F1: 0.8795 | Latency: 20.7s

Per-class F1 : [0.9781 0.9421 0.9465 0.6367 0.8941]

Per-class Precision: [0.9841 0.9504 0.9023 0.7335 0.8418]

Per-class Recall : [0.9721 0.9339 0.9954 0.5625 0.9532]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186047.41 173502.75 59992.7 47107.59 16881.45 18451.25

193044.09 43230.53 20140.18 17682.71]

Mean F1: 0.8547 | Latency: 20.6s

Per-class F1 : [0.9688 0.9258 0.9048 0.5766 0.8974]

Per-class Precision: [0.9892 0.9039 0.83 0.5424 0.8522]

Per-class Recall : [0.9493 0.9488 0.9945 0.6154 0.9477]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185565.81 173607.17 60107.46 47170.71 16946.49 18560.54

193099.69 43365.55 20510.39 17752.36]

Mean F1: 0.8466 | Latency: 20.6s

Per-class F1 : [0.9607 0.7935 0.9542 0.6242 0.9004]

Per-class Precision: [0.9844 0.6728 0.9225 0.6973 0.9 ]

Per-class Recall : [0.9382 0.9669 0.988 0.5649 0.9008]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185297.08 173638.31 60213.4 47149.57 16929.91 18492.89

192916.33 43308.16 20009.14 17809.63]

Mean F1: 0.9008 | Latency: 20.5s

Per-class F1 : [0.9826 0.9474 0.9595 0.69 0.9246]

Per-class Precision: [0.9853 0.9578 0.9284 0.7853 0.885 ]

Per-class Recall : [0.98 0.9372 0.9926 0.6154 0.9679]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186477.62 173761.66 60005.46 47303.97 16966.98 18482.84

192985.95 43261.15 20157.15 17671.23]

Mean F1: 0.8687 | Latency: 20.6s

Per-class F1 : [0.9724 0.94 0.9255 0.606 0.8999]

Per-class Precision: [0.9885 0.9354 0.8689 0.5629 0.8522]

Per-class Recall : [0.9568 0.9446 0.9899 0.6562 0.9532]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185563.32 173743.08 60462.94 47406.47 17166.52 18685.7

193075.43 43345.41 20278.49 18190.67]

Mean F1: 0.9038 | Latency: 20.6s

Per-class F1 : [0.9846 0.9505 0.96 0.6941 0.9297]

Per-class Precision: [0.9837 0.9651 0.9278 0.8083 0.9132]

Per-class Recall : [0.9855 0.9364 0.9945 0.6082 0.9467]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186066.75 173541.01 60259.84 47223.06 17054.39 18569.84

192907.77 43324.87 20233.7 17813.15]

Mean F1: 0.8902 | Latency: 20.5s

Per-class F1 : [0.9813 0.9504 0.9694 0.6408 0.9091]

Per-class Precision: [0.9828 0.9588 0.9611 0.7242 0.8636]

Per-class Recall : [0.9799 0.9421 0.9779 0.5745 0.9596]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185736.14 173576.7 60037.84 47278.11 16929.6 18508.9

193077.66 43359.18 20122.6 17695.61]

Mean F1: 0.8872 | Latency: 20.6s

Per-class F1 : [0.9799 0.9373 0.9305 0.6756 0.9126]

Per-class Precision: [0.9818 0.9828 0.8756 0.7598 0.8785]

Per-class Recall : [0.978 0.8959 0.9926 0.6082 0.9495]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185983.6 173957.9 60301.82 47476.16 17188.59 18694.73

193157. 43548.9 20514.86 18126.8 ]

Mean F1: 0.8488 | Latency: 20.6s

Per-class F1 : [0.9713 0.8896 0.938 0.5491 0.8959]

Per-class Precision: [0.9815 0.8273 0.915 0.6884 0.8524]

Per-class Recall : [0.9614 0.962 0.9622 0.4567 0.944 ]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186407.45 173764.68 60352.28 47381.28 17067.3 18612.72

193231.34 43443.19 20343.03 17805.87]

Mean F1: 0.8515 | Latency: 20.6s

Per-class F1 : [0.9566 0.9222 0.9337 0.6829 0.762 ]

Per-class Precision: [0.9886 0.9046 0.8865 0.7785 0.6231]

Per-class Recall : [0.9267 0.9405 0.9862 0.6082 0.9807]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185751.16 173722.59 60365.47 47467.6 17085.96 18820.82

193243.43 43608.34 20447.09 18003.39]

Mean F1: 0.8841 | Latency: 20.7s

Per-class F1 : [0.9734 0.9301 0.9753 0.698 0.8435]

Per-class Precision: [0.9866 0.92 0.9682 0.7903 0.7432]

Per-class Recall : [0.9606 0.9405 0.9825 0.625 0.9752]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186105.41 173856.17 60514.52 47507.89 17490.99 18932.79

193322.08 43720.87 20449.36 18966.85]

Mean F1: 0.8820 | Latency: 20.9s

Per-class F1 : [0.9752 0.938 0.9218 0.6816 0.8931]

Per-class Precision: [0.987 0.938 0.8596 0.7566 0.8325]

Per-class Recall : [0.9637 0.938 0.9935 0.6202 0.9633]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186749.3 173766.14 60407.08 47544.12 17227.99 18794.84

193152.53 43640.35 20425.29 18180.46]

Mean F1: 0.8693 | Latency: 20.7s

Per-class F1 : [0.9694 0.8937 0.9428 0.6574 0.8835]

Per-class Precision: [0.9893 0.8369 0.8962 0.6962 0.8173]

Per-class Recall : [0.9503 0.9587 0.9945 0.6226 0.9614]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185991.38 173953.03 60414.06 47712.89 17629.23 18951.12

193386.38 43855.84 20882.23 18337.46]

Mean F1: 0.8335 | Latency: 20.7s

Per-class F1 : [0.9461 0.8476 0.9387 0.6486 0.7866]

Per-class Precision: [0.9857 0.7703 0.8947 0.6658 0.6617]

Per-class Recall : [0.9096 0.9421 0.9871 0.6322 0.9697]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186681.86 174141.87 61073.51 48105.39 17628.91 19220.96

193600.04 44004.44 21104.29 19440.67]

Mean F1: 0.8994 | Latency: 20.8s

Per-class F1 : [0.982 0.9441 0.9545 0.6868 0.9294]

Per-class Precision: [0.9808 0.933 0.9161 0.9265 0.9215]

Per-class Recall : [0.9833 0.9554 0.9963 0.5457 0.9376]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186522.67 173847.03 60477. 47761.35 17235.13 19005.1

193356.27 43891.22 21724.37 18156.17]

Mean F1: 0.8970 | Latency: 20.8s

Per-class F1 : [0.9806 0.9563 0.9395 0.6788 0.9297]

Per-class Precision: [0.9865 0.9623 0.8903 0.7394 0.8961]

Per-class Recall : [0.9747 0.9504 0.9945 0.6274 0.966 ]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185928.62 173840.79 60470.6 47467.15 17283.37 18740.62

193312.2 43545.58 20375.54 17948.62]

Mean F1: 0.8750 | Latency: 20.7s

Per-class F1 : [0.9732 0.8975 0.9664 0.6486 0.8892]

Per-class Precision: [0.9833 0.8645 0.9414 0.7445 0.8298]

Per-class Recall : [0.9633 0.9331 0.9926 0.5745 0.9578]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186284.89 174092.38 60529.41 48150.99 17510.78 19008.92

193534.29 43782.39 21022.48 18953.5 ]

Mean F1: 0.8859 | Latency: 20.8s

Per-class F1 : [0.9786 0.9492 0.9557 0.6492 0.8966]

Per-class Precision: [0.9809 0.9564 0.9294 0.763 0.8538]

Per-class Recall : [0.9763 0.9421 0.9834 0.5649 0.944 ]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186196.22 173816.8 60751.18 47655.57 17561.46 18894.39

193583.87 43627.29 20500.49 18712.89]

Mean F1: 0.8330 | Latency: 20.6s

Per-class F1 : [0.9536 0.8604 0.9515 0.6162 0.7833]

Per-class Precision: [0.9855 0.7952 0.9366 0.7203 0.6505]

Per-class Recall : [0.9236 0.9372 0.9668 0.5385 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185948.62 174151.79 60581.39 47528.43 17369.74 18909.05

193280.35 43615.16 20594.6 18450.09]

Mean F1: 0.8952 | Latency: 20.7s

Per-class F1 : [0.9813 0.9519 0.9396 0.6964 0.9065]

Per-class Precision: [0.9892 0.9551 0.8883 0.8278 0.8469]

Per-class Recall : [0.9737 0.9488 0.9972 0.601 0.9752]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2185916.01 174037.61 60590.84 48350.81 17638.06 19132.6

193696.24 43987.05 21117.45 18784.87]

Mean F1: 0.8986 | Latency: 20.8s

Per-class F1 : [0.9843 0.9593 0.9682 0.6775 0.9035]

Per-class Precision: [0.9848 0.9852 0.9424 0.8745 0.8444]

Per-class Recall : [0.9839 0.9347 0.9954 0.5529 0.9715]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186043.33 174083.16 60513.28 47678.38 17146.25 18743.18

193439.27 43495.3 20416.51 18117.8 ]

Mean F1: 0.9042 | Latency: 20.7s

Per-class F1 : [0.9844 0.9627 0.9462 0.7083 0.9193]

Per-class Precision: [0.9859 0.9778 0.9002 0.9297 0.8754]

Per-class Recall : [0.9828 0.9479 0.9972 0.5721 0.9679]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186241.73 174208.06 60549.22 47771.72 17349.64 18854.95

193411.55 43735.58 20656.28 18472.72]

Mean F1: 0.9008 | Latency: 20.6s

Per-class F1 : [0.9845 0.9678 0.9651 0.6575 0.9292]

Per-class Precision: [0.9819 0.9805 0.9374 0.7685 0.9245]

Per-class Recall : [0.9871 0.9554 0.9945 0.5745 0.9339]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186168.97 174235.09 60588.54 47727.88 17684.04 18916.81

193557.89 43952.73 20566.35 18853.92]

Mean F1: 0.9086 | Latency: 20.5s

Per-class F1 : [0.9848 0.9578 0.9716 0.7106 0.9181]

Per-class Precision: [0.9862 0.9695 0.9522 0.8275 0.8732]

Per-class Recall : [0.9834 0.9463 0.9917 0.6226 0.9679]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [B_Krum] ---

[Krum] Winner: client 5 | Scores: [2186536.55 174132.34 60717.99 47854.12 17444.72 18846.12

193553.37 43744.2 20742.83 18298.17]

Mean F1: 0.8923 | Latency: 20.6s

Per-class F1 : [0.9803 0.9577 0.9682 0.6456 0.9095]

Per-class Precision: [0.9867 0.9609 0.9535 0.6543 0.8607]

Per-class Recall : [0.9739 0.9545 0.9834 0.637 0.9642]

Support : [11237 1210 1085 416 1089]



[B_Krum] Final Mean F1: 0.8923



======================================================================

METHOD: C_MultiKrum | ATTACK: label_flip

======================================================================



--- Round 1/40 [C_MultiKrum] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.1332 | Latency: 20.8s

Per-class F1 : [0.2386 0.2373 0. 0.0203 0.1696]

Per-class Precision: [0.7961 0.1407 0. 0.0343 0.0993]

Per-class Recall : [0.1403 0.7587 0. 0.0144 0.5794]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.5005 | Latency: 20.7s

Per-class F1 : [0.8309 0.579 0.3311 0.0893 0.6722]

Per-class Precision: [0.887 0.4513 0.3674 0.0896 0.5554]

Per-class Recall : [0.7814 0.8074 0.3014 0.0889 0.8512]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.3371 | Latency: 20.7s

Per-class F1 : [0.6408 0.2858 0.2861 0.1494 0.3232]

Per-class Precision: [0.9399 0.197 0.7159 0.1857 0.1937]

Per-class Recall : [0.4861 0.5207 0.1788 0.125 0.9743]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.4430 | Latency: 20.8s

Per-class F1 : [0.8817 0.3603 0.2415 0.1871 0.5444]

Per-class Precision: [0.95 0.2866 0.8736 0.233 0.3779]

Per-class Recall : [0.8226 0.4851 0.1401 0.1562 0.9734]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.5845 | Latency: 20.7s

Per-class F1 : [0.898 0.6432 0.4754 0.2982 0.6077]

Per-class Precision: [0.9453 0.5381 0.9581 0.3806 0.439 ]

Per-class Recall : [0.8551 0.7992 0.3161 0.2452 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.6876 | Latency: 20.6s

Per-class F1 : [0.9196 0.6924 0.6919 0.4592 0.675 ]

Per-class Precision: [0.9639 0.6246 0.9622 0.3824 0.5168]

Per-class Recall : [0.8791 0.7769 0.5401 0.5745 0.9725]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8135 | Latency: 20.8s

Per-class F1 : [0.9471 0.8846 0.8429 0.5114 0.8816]

Per-class Precision: [0.9639 0.8349 0.9804 0.394 0.8298]

Per-class Recall : [0.9309 0.9405 0.7392 0.7284 0.9403]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8356 | Latency: 20.7s

Per-class F1 : [0.9566 0.917 0.8901 0.5684 0.8461]

Per-class Precision: [0.9737 0.9118 0.9716 0.4722 0.7477]

Per-class Recall : [0.94 0.9223 0.8212 0.7139 0.9743]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8464 | Latency: 20.7s

Per-class F1 : [0.9671 0.8966 0.867 0.6246 0.8768]

Per-class Precision: [0.9755 0.8643 0.9692 0.5881 0.7952]

Per-class Recall : [0.9589 0.9314 0.7843 0.6659 0.977 ]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.7703 | Latency: 20.7s

Per-class F1 : [0.9359 0.7894 0.856 0.5467 0.7233]

Per-class Precision: [0.9631 0.8825 0.9775 0.4318 0.5717]

Per-class Recall : [0.9102 0.714 0.7613 0.7452 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.7724 | Latency: 20.8s

Per-class F1 : [0.9452 0.8317 0.7456 0.5676 0.772 ]

Per-class Precision: [0.9722 0.7461 0.9747 0.4946 0.6404]

Per-class Recall : [0.9196 0.9397 0.6037 0.6659 0.9715]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.7971 | Latency: 20.7s

Per-class F1 : [0.94 0.8345 0.9286 0.6114 0.6708]

Per-class Precision: [0.9804 0.9308 0.9815 0.5184 0.5066]

Per-class Recall : [0.9028 0.7562 0.8811 0.7452 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8623 | Latency: 20.7s

Per-class F1 : [0.9643 0.9223 0.9621 0.5957 0.8673]

Per-class Precision: [0.9821 0.9598 0.9781 0.487 0.7769]

Per-class Recall : [0.9471 0.8876 0.9465 0.7668 0.9816]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8585 | Latency: 20.8s

Per-class F1 : [0.9567 0.9431 0.9589 0.6449 0.7891]

Per-class Precision: [0.9857 0.9415 0.9845 0.5603 0.6577]

Per-class Recall : [0.9294 0.9446 0.9346 0.7596 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8683 | Latency: 20.7s

Per-class F1 : [0.9607 0.962 0.9776 0.6154 0.8256]

Per-class Precision: [0.9903 0.973 0.9896 0.4935 0.7086]

Per-class Recall : [0.9329 0.9512 0.9659 0.8173 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8893 | Latency: 20.9s

Per-class F1 : [0.9706 0.9581 0.9792 0.7071 0.8317]

Per-class Precision: [0.9892 0.9826 0.9815 0.6632 0.7162]

Per-class Recall : [0.9527 0.9347 0.977 0.7572 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8982 | Latency: 20.8s

Per-class F1 : [0.9749 0.9435 0.9723 0.7357 0.8647]

Per-class Precision: [0.9909 0.9226 0.9904 0.7133 0.7682]

Per-class Recall : [0.9594 0.9653 0.9548 0.7596 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9048 | Latency: 20.7s

Per-class F1 : [0.9767 0.9707 0.9772 0.719 0.8805]

Per-class Precision: [0.9914 0.9695 0.9859 0.6574 0.7964]

Per-class Recall : [0.9625 0.9719 0.9687 0.7933 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9198 | Latency: 20.8s

Per-class F1 : [0.982 0.9726 0.9834 0.7475 0.9135]

Per-class Precision: [0.991 0.9783 0.9861 0.6957 0.8562]

Per-class Recall : [0.9731 0.9669 0.9806 0.8077 0.9789]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8896 | Latency: 20.7s

Per-class F1 : [0.9691 0.9762 0.9805 0.6699 0.8525]

Per-class Precision: [0.9917 0.9849 0.9888 0.5597 0.7518]

Per-class Recall : [0.9474 0.9678 0.9724 0.8341 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9109 | Latency: 20.7s

Per-class F1 : [0.9789 0.9667 0.9842 0.7076 0.9173]

Per-class Precision: [0.9901 0.9879 0.9897 0.6157 0.856 ]

Per-class Recall : [0.968 0.9463 0.9788 0.8317 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8929 | Latency: 20.8s

Per-class F1 : [0.9715 0.9743 0.9805 0.6479 0.8906]

Per-class Precision: [0.9909 0.9775 0.9897 0.5307 0.8143]

Per-class Recall : [0.9527 0.9711 0.9714 0.8317 0.9826]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8785 | Latency: 20.7s

Per-class F1 : [0.9638 0.9682 0.978 0.6698 0.8125]

Per-class Precision: [0.9913 0.9956 0.9924 0.5554 0.6867]

Per-class Recall : [0.9377 0.9421 0.9641 0.8438 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9242 | Latency: 20.8s

Per-class F1 : [0.9832 0.9749 0.9742 0.7586 0.9303]

Per-class Precision: [0.9883 0.9881 0.9905 0.706 0.8855]

Per-class Recall : [0.9781 0.962 0.9585 0.8197 0.9798]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9200 | Latency: 20.7s

Per-class F1 : [0.9807 0.9724 0.9866 0.7368 0.9235]

Per-class Precision: [0.993 0.9681 0.988 0.6554 0.8682]

Per-class Recall : [0.9688 0.9769 0.9853 0.8413 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8906 | Latency: 20.6s

Per-class F1 : [0.9676 0.9681 0.9655 0.7588 0.7929]

Per-class Precision: [0.991 0.9846 0.9903 0.7029 0.6613]

Per-class Recall : [0.9452 0.9521 0.9419 0.8245 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9298 | Latency: 20.7s

Per-class F1 : [0.9838 0.9768 0.9839 0.7872 0.9172]

Per-class Precision: [0.9936 0.9784 0.9825 0.7511 0.8531]

Per-class Recall : [0.9743 0.9752 0.9853 0.8269 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9090 | Latency: 20.6s

Per-class F1 : [0.9767 0.9707 0.9801 0.7432 0.8745]

Per-class Precision: [0.9936 0.968 0.9824 0.6814 0.782 ]

Per-class Recall : [0.9603 0.9736 0.9779 0.8173 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9344 | Latency: 20.7s

Per-class F1 : [0.9871 0.9796 0.9793 0.7848 0.9413]

Per-class Precision: [0.9904 0.9882 0.9771 0.7985 0.8996]

Per-class Recall : [0.9837 0.9711 0.9816 0.7716 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9209 | Latency: 20.9s

Per-class F1 : [0.982 0.9779 0.9852 0.7233 0.9361]

Per-class Precision: [0.9918 0.9874 0.9861 0.6413 0.8886]

Per-class Recall : [0.9724 0.9686 0.9843 0.8293 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9095 | Latency: 20.6s

Per-class F1 : [0.9775 0.9808 0.98 0.7112 0.8977]

Per-class Precision: [0.9928 0.9907 0.9869 0.6147 0.8205]

Per-class Recall : [0.9626 0.9711 0.9733 0.8438 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9342 | Latency: 20.7s

Per-class F1 : [0.9859 0.9797 0.9833 0.7625 0.9597]

Per-class Precision: [0.992 0.9833 0.9906 0.6845 0.937 ]

Per-class Recall : [0.98 0.976 0.976 0.8606 0.9835]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9231 | Latency: 20.7s

Per-class F1 : [0.9835 0.9783 0.9866 0.7237 0.9435]

Per-class Precision: [0.9927 0.989 0.9907 0.6336 0.902 ]

Per-class Recall : [0.9744 0.9678 0.9825 0.8438 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.8975 | Latency: 20.6s

Per-class F1 : [0.9736 0.978 0.9837 0.6598 0.8922]

Per-class Precision: [0.994 0.9833 0.9925 0.5388 0.8108]

Per-class Recall : [0.954 0.9727 0.9751 0.851 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9239 | Latency: 20.7s

Per-class F1 : [0.983 0.9643 0.9852 0.7677 0.919 ]

Per-class Precision: [0.9915 0.9913 0.9907 0.6946 0.8576]

Per-class Recall : [0.9747 0.9388 0.9797 0.8582 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9378 | Latency: 20.9s

Per-class F1 : [0.9857 0.9766 0.9903 0.7891 0.9472]

Per-class Precision: [0.9945 0.9698 0.9899 0.7234 0.9089]

Per-class Recall : [0.977 0.9835 0.9908 0.8678 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9432 | Latency: 20.7s

Per-class F1 : [0.9881 0.98 0.9898 0.796 0.962 ]

Per-class Precision: [0.9929 0.9858 0.9926 0.7387 0.9373]

Per-class Recall : [0.9833 0.9744 0.9871 0.863 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9222 | Latency: 20.7s

Per-class F1 : [0.9813 0.9798 0.988 0.7123 0.9495]

Per-class Precision: [0.9941 0.9778 0.9926 0.6007 0.9099]

Per-class Recall : [0.9689 0.9818 0.9834 0.875 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9343 | Latency: 20.7s

Per-class F1 : [0.9857 0.9834 0.9899 0.7443 0.9681]

Per-class Precision: [0.9935 0.9859 0.9908 0.6516 0.9489]

Per-class Recall : [0.978 0.981 0.9889 0.8678 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [C_MultiKrum] ---

[Multi-Krum k=7] Selected clients: [np.int64(5), np.int64(10), np.int64(6), np.int64(9), np.int64(8), np.int64(4), np.int64(3)] | Rejected: [1, 2, 7]

Mean F1: 0.9186 | Latency: 20.7s

Per-class F1 : [0.9805 0.9808 0.9912 0.6929 0.9477]

Per-class Precision: [0.9941 0.9916 0.9944 0.5675 0.9082]

Per-class Recall : [0.9673 0.9702 0.988 0.8894 0.9908]

Support : [11237 1210 1085 416 1089]



[C_MultiKrum] Final Mean F1: 0.9186



======================================================================

METHOD: D_Median | ATTACK: label_flip

======================================================================



--- Round 1/40 [D_Median] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.1222 | Latency: 20.7s

Per-class F1 : [0.1878 0.2477 0. 0. 0.1756]

Per-class Precision: [0.7979 0.1582 0. 0. 0.0983]

Per-class Recall : [0.1064 0.5702 0. 0. 0.8228]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.3723 | Latency: 20.8s

Per-class F1 : [0.6562 0.3774 0.1148 0.126 0.587 ]

Per-class Precision: [0.8983 0.2364 0.1119 0.1026 0.4554]

Per-class Recall : [0.517 0.9355 0.118 0.1635 0.8255]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.3405 | Latency: 20.8s

Per-class F1 : [0.6807 0.3533 0.0138 0.1813 0.4736]

Per-class Precision: [0.9286 0.2199 0.1067 0.2 0.3181]

Per-class Recall : [0.5372 0.8983 0.0074 0.1659 0.9265]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.6615 | Latency: 20.8s

Per-class F1 : [0.9042 0.6651 0.7685 0.1973 0.7725]

Per-class Precision: [0.9845 0.513 0.7084 0.1867 0.6642]

Per-class Recall : [0.836 0.9455 0.8396 0.2091 0.9229]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.4892 | Latency: 20.9s

Per-class F1 : [0.822 0.5692 0.3526 0.2141 0.4881]

Per-class Precision: [0.9827 0.4061 0.61 0.198 0.3273]

Per-class Recall : [0.7065 0.9512 0.2479 0.2332 0.9596]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.6660 | Latency: 20.8s

Per-class F1 : [0.9166 0.748 0.726 0.2484 0.6907]

Per-class Precision: [0.9876 0.6192 0.7934 0.2182 0.5338]

Per-class Recall : [0.8552 0.9446 0.6691 0.2885 0.978 ]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.6100 | Latency: 20.8s

Per-class F1 : [0.8981 0.6852 0.3799 0.4982 0.5888]

Per-class Precision: [0.9888 0.5371 0.8656 0.4006 0.4193]

Per-class Recall : [0.8227 0.9463 0.2433 0.6587 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.7861 | Latency: 20.8s

Per-class F1 : [0.9397 0.8414 0.8638 0.5547 0.7308]

Per-class Precision: [0.991 0.7502 0.9502 0.4617 0.5827]

Per-class Recall : [0.8935 0.9579 0.7917 0.6947 0.9798]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8058 | Latency: 20.9s

Per-class F1 : [0.9303 0.9269 0.9482 0.5521 0.6717]

Per-class Precision: [0.9917 0.9061 0.9517 0.453 0.5085]

Per-class Recall : [0.876 0.9488 0.9447 0.7067 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.7897 | Latency: 20.9s

Per-class F1 : [0.9371 0.874 0.8726 0.5992 0.6656]

Per-class Precision: [0.9899 0.8421 0.9664 0.5137 0.5007]

Per-class Recall : [0.8896 0.9083 0.7954 0.7188 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8219 | Latency: 20.8s

Per-class F1 : [0.9592 0.8472 0.8638 0.5774 0.862 ]

Per-class Precision: [0.9907 0.7445 0.9584 0.4794 0.7724]

Per-class Recall : [0.9297 0.9826 0.7862 0.726 0.9752]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8659 | Latency: 20.7s

Per-class F1 : [0.9602 0.9532 0.9671 0.6536 0.7953]

Per-class Precision: [0.9909 0.9463 0.9729 0.5722 0.6646]

Per-class Recall : [0.9313 0.9603 0.9613 0.762 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8678 | Latency: 20.8s

Per-class F1 : [0.9622 0.9127 0.9632 0.6653 0.8356]

Per-class Precision: [0.9923 0.8533 0.9855 0.5754 0.7283]

Per-class Recall : [0.9338 0.981 0.9419 0.7885 0.9798]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8391 | Latency: 20.8s

Per-class F1 : [0.9542 0.9183 0.9239 0.621 0.7782]

Per-class Precision: [0.9914 0.8716 0.9813 0.5142 0.6435]

Per-class Recall : [0.9197 0.9702 0.8728 0.7837 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9013 | Latency: 20.8s

Per-class F1 : [0.9766 0.9508 0.9664 0.7293 0.8834]

Per-class Precision: [0.9919 0.929 0.9783 0.6895 0.7988]

Per-class Recall : [0.9618 0.9736 0.9548 0.774 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8992 | Latency: 20.8s

Per-class F1 : [0.9756 0.966 0.9711 0.6824 0.9009]

Per-class Precision: [0.9915 0.9684 0.9821 0.5786 0.8299]

Per-class Recall : [0.9601 0.9636 0.9604 0.8317 0.9853]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8978 | Latency: 20.8s

Per-class F1 : [0.971 0.9499 0.9739 0.7491 0.8451]

Per-class Precision: [0.9916 0.9228 0.984 0.7097 0.7403]

Per-class Recall : [0.9511 0.9785 0.9641 0.7933 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8970 | Latency: 20.7s

Per-class F1 : [0.9721 0.9632 0.9764 0.717 0.8563]

Per-class Precision: [0.9935 0.9523 0.9787 0.6431 0.754 ]

Per-class Recall : [0.9517 0.9744 0.9742 0.8101 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.8777 | Latency: 20.9s

Per-class F1 : [0.9744 0.9323 0.8959 0.7532 0.8325]

Per-class Precision: [0.9915 0.8945 0.9803 0.7209 0.7193]

Per-class Recall : [0.9579 0.9736 0.8249 0.7885 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9145 | Latency: 20.6s

Per-class F1 : [0.9788 0.9401 0.9764 0.783 0.894 ]

Per-class Precision: [0.9939 0.9045 0.9796 0.7685 0.8169]

Per-class Recall : [0.9642 0.9785 0.9733 0.7981 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9175 | Latency: 20.8s

Per-class F1 : [0.9819 0.9752 0.9838 0.7128 0.9337]

Per-class Precision: [0.9942 0.976 0.9861 0.6184 0.8836]

Per-class Recall : [0.9699 0.9744 0.9816 0.8413 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9317 | Latency: 20.7s

Per-class F1 : [0.9848 0.9772 0.9816 0.7944 0.9206]

Per-class Precision: [0.9929 0.9808 0.9816 0.7727 0.8598]

Per-class Recall : [0.9769 0.9736 0.9816 0.8173 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9256 | Latency: 20.8s

Per-class F1 : [0.983 0.9666 0.9814 0.7682 0.9287]

Per-class Precision: [0.9936 0.9541 0.9888 0.7018 0.8775]

Per-class Recall : [0.9726 0.9793 0.9742 0.8486 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9344 | Latency: 20.7s

Per-class F1 : [0.985 0.972 0.9866 0.7826 0.9456]

Per-class Precision: [0.9938 0.9688 0.9871 0.7143 0.9051]

Per-class Recall : [0.9763 0.9752 0.9862 0.8654 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9337 | Latency: 20.8s

Per-class F1 : [0.9854 0.9636 0.9834 0.778 0.958 ]

Per-class Precision: [0.9938 0.943 0.9852 0.7166 0.933 ]

Per-class Recall : [0.9771 0.9851 0.9816 0.851 0.9844]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9336 | Latency: 20.7s

Per-class F1 : [0.9849 0.9821 0.9766 0.7852 0.939 ]

Per-class Precision: [0.9919 0.9899 0.9905 0.7154 0.8938]

Per-class Recall : [0.978 0.9744 0.9631 0.8702 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9209 | Latency: 20.7s

Per-class F1 : [0.9816 0.9748 0.9824 0.7461 0.9196]

Per-class Precision: [0.9945 0.9736 0.9879 0.6599 0.8566]

Per-class Recall : [0.9689 0.976 0.977 0.8582 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9359 | Latency: 20.7s

Per-class F1 : [0.9866 0.9758 0.9847 0.7772 0.9553]

Per-class Precision: [0.9951 0.9667 0.9888 0.7037 0.9229]

Per-class Recall : [0.9783 0.9851 0.9806 0.8678 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9215 | Latency: 20.6s

Per-class F1 : [0.9802 0.9746 0.988 0.7367 0.9282]

Per-class Precision: [0.996 0.9667 0.988 0.6229 0.873 ]

Per-class Recall : [0.9648 0.9826 0.988 0.9014 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9501 | Latency: 20.8s

Per-class F1 : [0.9896 0.9823 0.9898 0.8192 0.9694]

Per-class Precision: [0.9946 0.9795 0.9917 0.7646 0.9498]

Per-class Recall : [0.9846 0.9851 0.988 0.8822 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9379 | Latency: 20.7s

Per-class F1 : [0.9868 0.9791 0.9885 0.7725 0.9629]

Per-class Precision: [0.9951 0.9731 0.988 0.69 0.9389]

Per-class Recall : [0.9786 0.9851 0.9889 0.8774 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9441 | Latency: 20.7s

Per-class F1 : [0.9879 0.9732 0.9885 0.8359 0.9352]

Per-class Precision: [0.9947 0.9727 0.988 0.8213 0.8833]

Per-class Recall : [0.9811 0.9736 0.9889 0.851 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9430 | Latency: 20.9s

Per-class F1 : [0.9883 0.9706 0.9819 0.8164 0.9579]

Per-class Precision: [0.9946 0.9589 0.987 0.7766 0.9262]

Per-class Recall : [0.9821 0.9826 0.977 0.8606 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9494 | Latency: 20.8s

Per-class F1 : [0.9895 0.9807 0.9903 0.8275 0.9589]

Per-class Precision: [0.9944 0.9739 0.9881 0.8032 0.9331]

Per-class Recall : [0.9847 0.9876 0.9926 0.8534 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9394 | Latency: 20.8s

Per-class F1 : [0.9867 0.9794 0.988 0.7974 0.9457]

Per-class Precision: [0.9957 0.9754 0.9898 0.7258 0.9044]

Per-class Recall : [0.9778 0.9835 0.9862 0.8846 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9345 | Latency: 20.9s

Per-class F1 : [0.9873 0.9555 0.9684 0.8058 0.9556]

Per-class Precision: [0.9944 0.9283 0.9932 0.7521 0.9245]

Per-class Recall : [0.9802 0.9843 0.9447 0.8678 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9442 | Latency: 20.9s

Per-class F1 : [0.9867 0.9692 0.989 0.8175 0.9588]

Per-class Precision: [0.9955 0.9507 0.9854 0.7652 0.9264]

Per-class Recall : [0.9781 0.9884 0.9926 0.8774 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9463 | Latency: 20.8s

Per-class F1 : [0.9883 0.9718 0.9903 0.8161 0.9651]

Per-class Precision: [0.9947 0.9605 0.9908 0.7647 0.9415]

Per-class Recall : [0.9819 0.9835 0.9899 0.875 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9482 | Latency: 20.9s

Per-class F1 : [0.9889 0.9823 0.9926 0.8151 0.9622]

Per-class Precision: [0.9955 0.9802 0.9917 0.7556 0.9335]

Per-class Recall : [0.9825 0.9843 0.9935 0.8846 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [D_Median] ---

[Median] Aggregated 10 clients (coordinate-wise median)

Mean F1: 0.9475 | Latency: 20.9s

Per-class F1 : [0.9885 0.9766 0.9904 0.8198 0.9621]

Per-class Precision: [0.9951 0.9691 0.9863 0.7712 0.935 ]

Per-class Recall : [0.9819 0.9843 0.9945 0.875 0.9908]

Support : [11237 1210 1085 416 1089]



[D_Median] Final Mean F1: 0.9475



======================================================================

METHOD: E_TrimmedMean | ATTACK: label_flip

======================================================================



--- Round 1/40 [E_TrimmedMean] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.1073 | Latency: 20.9s

Per-class F1 : [0.1298 0.2442 0. 0. 0.1623]

Per-class Precision: [0.8233 0.1537 0. 0. 0.0906]

Per-class Recall : [0.0705 0.5934 0. 0. 0.7796]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.5079 | Latency: 20.9s

Per-class F1 : [0.7763 0.5061 0.3918 0.1699 0.6953]

Per-class Precision: [0.927 0.3583 0.3283 0.1353 0.5669]

Per-class Recall : [0.6678 0.8612 0.4857 0.2284 0.899 ]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.4899 | Latency: 20.8s

Per-class F1 : [0.8476 0.5196 0.3454 0.177 0.56 ]

Per-class Precision: [0.9665 0.3683 0.6738 0.1965 0.3951]

Per-class Recall : [0.7547 0.8818 0.2323 0.1611 0.9614]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.6915 | Latency: 20.9s

Per-class F1 : [0.9334 0.7749 0.8103 0.2234 0.7157]

Per-class Precision: [0.9787 0.6867 0.8435 0.2397 0.5671]

Per-class Recall : [0.8921 0.8893 0.7797 0.2091 0.9697]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.6154 | Latency: 20.9s

Per-class F1 : [0.8901 0.743 0.6533 0.2294 0.5613]

Per-class Precision: [0.9706 0.6271 0.9011 0.2193 0.3943]

Per-class Recall : [0.822 0.9116 0.5124 0.2404 0.9734]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.7860 | Latency: 21.0s

Per-class F1 : [0.9455 0.8684 0.86 0.4723 0.784 ]

Per-class Precision: [0.9871 0.793 0.9478 0.3689 0.6579]

Per-class Recall : [0.9072 0.9595 0.7871 0.6562 0.9697]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8049 | Latency: 21.0s

Per-class F1 : [0.9454 0.8267 0.8913 0.5806 0.7806]

Per-class Precision: [0.9895 0.7281 0.937 0.5 0.6544]

Per-class Recall : [0.9051 0.9562 0.8498 0.6923 0.9669]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.7901 | Latency: 20.9s

Per-class F1 : [0.9397 0.8701 0.8621 0.5927 0.6861]

Per-class Precision: [0.9868 0.8282 0.9756 0.5042 0.5268]

Per-class Recall : [0.8969 0.9165 0.7724 0.7188 0.9835]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8769 | Latency: 20.9s

Per-class F1 : [0.9699 0.9561 0.9652 0.6398 0.8534]

Per-class Precision: [0.9906 0.9487 0.9728 0.5618 0.7553]

Per-class Recall : [0.95 0.9636 0.9576 0.7428 0.9807]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8379 | Latency: 20.9s

Per-class F1 : [0.9662 0.8865 0.878 0.6134 0.8454]

Per-class Precision: [0.9887 0.8269 0.9742 0.5201 0.7482]

Per-class Recall : [0.9447 0.9554 0.7991 0.7476 0.9715]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8830 | Latency: 20.9s

Per-class F1 : [0.9712 0.9552 0.9651 0.6645 0.859 ]

Per-class Precision: [0.9924 0.9344 0.9616 0.5951 0.767 ]

Per-class Recall : [0.9509 0.9769 0.9687 0.7524 0.9761]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8636 | Latency: 20.9s

Per-class F1 : [0.9603 0.9536 0.9664 0.6449 0.7931]

Per-class Precision: [0.9918 0.9733 0.9801 0.5275 0.6628]

Per-class Recall : [0.9307 0.9347 0.953 0.8293 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8666 | Latency: 20.9s

Per-class F1 : [0.9639 0.9197 0.931 0.7299 0.7886]

Per-class Precision: [0.9908 0.8702 0.9767 0.7196 0.6591]

Per-class Recall : [0.9384 0.9752 0.8894 0.7404 0.9816]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8785 | Latency: 21.0s

Per-class F1 : [0.9633 0.9744 0.9661 0.7054 0.7832]

Per-class Precision: [0.9932 0.9736 0.9866 0.6204 0.6492]

Per-class Recall : [0.9351 0.9752 0.9465 0.8173 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8537 | Latency: 21.1s

Per-class F1 : [0.9521 0.9481 0.9435 0.701 0.7239]

Per-class Precision: [0.989 0.9314 0.9919 0.6562 0.5703]

Per-class Recall : [0.9179 0.9653 0.8995 0.7524 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9186 | Latency: 20.8s

Per-class F1 : [0.9804 0.972 0.9824 0.7511 0.907 ]

Per-class Precision: [0.9928 0.9688 0.9879 0.6831 0.8403]

Per-class Recall : [0.9684 0.9752 0.977 0.8341 0.9853]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8872 | Latency: 21.0s

Per-class F1 : [0.963 0.9609 0.9758 0.7561 0.7804]

Per-class Precision: [0.993 0.947 0.9868 0.7016 0.6453]

Per-class Recall : [0.9348 0.9752 0.965 0.8197 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8769 | Latency: 20.9s

Per-class F1 : [0.9573 0.9617 0.9762 0.7288 0.7604]

Per-class Precision: [0.9942 0.9486 0.9905 0.64 0.6158]

Per-class Recall : [0.923 0.9752 0.9622 0.8462 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8866 | Latency: 20.9s

Per-class F1 : [0.9625 0.9523 0.9694 0.7831 0.7656]

Per-class Precision: [0.9925 0.9397 0.9904 0.7643 0.6224]

Per-class Recall : [0.9342 0.9653 0.9493 0.8029 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9237 | Latency: 20.9s

Per-class F1 : [0.9808 0.976 0.9837 0.7863 0.8918]

Per-class Precision: [0.9938 0.9776 0.9934 0.729 0.8102]

Per-class Recall : [0.9681 0.9744 0.9742 0.8534 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9148 | Latency: 20.9s

Per-class F1 : [0.9785 0.966 0.9708 0.7734 0.8854]

Per-class Precision: [0.9928 0.9578 0.9923 0.7072 0.8009]

Per-class Recall : [0.9647 0.9744 0.9502 0.8534 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9108 | Latency: 21.0s

Per-class F1 : [0.9766 0.9481 0.9747 0.7666 0.888 ]

Per-class Precision: [0.9936 0.9167 0.9914 0.699 0.8075]

Per-class Recall : [0.96 0.9818 0.9585 0.8486 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9002 | Latency: 20.9s

Per-class F1 : [0.9698 0.9552 0.9771 0.777 0.8221]

Per-class Precision: [0.9943 0.9337 0.9915 0.7309 0.7003]

Per-class Recall : [0.9464 0.9777 0.9631 0.8293 0.9954]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.8968 | Latency: 20.9s

Per-class F1 : [0.9702 0.9318 0.9581 0.794 0.8301]

Per-class Precision: [0.9939 0.8866 0.9921 0.7616 0.7128]

Per-class Recall : [0.9476 0.9818 0.9263 0.8293 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9326 | Latency: 20.9s

Per-class F1 : [0.984 0.9777 0.9856 0.7917 0.9239]

Per-class Precision: [0.9945 0.9761 0.9907 0.7278 0.8641]

Per-class Recall : [0.9737 0.9793 0.9806 0.8678 0.9927]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9154 | Latency: 20.8s

Per-class F1 : [0.9775 0.9686 0.9833 0.7664 0.8816]

Per-class Precision: [0.9942 0.9572 0.9925 0.702 0.7917]

Per-class Recall : [0.9613 0.9802 0.9742 0.8438 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9286 | Latency: 20.9s

Per-class F1 : [0.9828 0.9674 0.9833 0.7955 0.9139]

Per-class Precision: [0.9946 0.9542 0.9888 0.7505 0.846 ]

Per-class Recall : [0.9712 0.981 0.9779 0.8462 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9129 | Latency: 20.9s

Per-class F1 : [0.9772 0.9439 0.96 0.82 0.8633]

Per-class Precision: [0.9943 0.9096 0.9921 0.7933 0.7643]

Per-class Recall : [0.9607 0.981 0.93 0.8486 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9275 | Latency: 20.8s

Per-class F1 : [0.9811 0.9826 0.9884 0.7849 0.9002]

Per-class Precision: [0.9948 0.985 0.9953 0.7101 0.8223]

Per-class Recall : [0.9678 0.9802 0.9816 0.8774 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9367 | Latency: 20.9s

Per-class F1 : [0.9843 0.9746 0.988 0.8143 0.9221]

Per-class Precision: [0.9954 0.9659 0.9916 0.7615 0.8595]

Per-class Recall : [0.9735 0.9835 0.9843 0.875 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9257 | Latency: 20.9s

Per-class F1 : [0.9796 0.9718 0.9865 0.8146 0.8762]

Per-class Precision: [0.995 0.9619 0.9944 0.7773 0.7831]

Per-class Recall : [0.9646 0.9818 0.9788 0.8558 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9322 | Latency: 21.0s

Per-class F1 : [0.9826 0.9742 0.9846 0.8102 0.9092]

Per-class Precision: [0.9947 0.9659 0.9944 0.7526 0.8381]

Per-class Recall : [0.9707 0.9826 0.9751 0.8774 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9298 | Latency: 20.9s

Per-class F1 : [0.9823 0.9719 0.9875 0.7764 0.9312]

Per-class Precision: [0.9952 0.9582 0.9926 0.6917 0.8761]

Per-class Recall : [0.9698 0.986 0.9825 0.8846 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9253 | Latency: 20.9s

Per-class F1 : [0.9782 0.9774 0.9898 0.797 0.8838]

Per-class Precision: [0.9955 0.9707 0.9935 0.7204 0.7947]

Per-class Recall : [0.9616 0.9843 0.9862 0.8918 0.9954]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9430 | Latency: 20.9s

Per-class F1 : [0.9872 0.9763 0.9885 0.8214 0.9415]

Per-class Precision: [0.9952 0.9668 0.9907 0.7797 0.8969]

Per-class Recall : [0.9793 0.986 0.9862 0.8678 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9445 | Latency: 20.9s

Per-class F1 : [0.9874 0.9839 0.9889 0.8142 0.948 ]

Per-class Precision: [0.9953 0.9819 0.9935 0.7541 0.9042]

Per-class Recall : [0.9795 0.986 0.9843 0.8846 0.9963]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9395 | Latency: 20.9s

Per-class F1 : [0.9857 0.9802 0.9889 0.8057 0.9368]

Per-class Precision: [0.9955 0.9762 0.9926 0.738 0.8862]

Per-class Recall : [0.9762 0.9843 0.9853 0.887 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9446 | Latency: 20.9s

Per-class F1 : [0.9866 0.9832 0.9903 0.8242 0.9388]

Per-class Precision: [0.9955 0.9771 0.9926 0.7715 0.8898]

Per-class Recall : [0.9779 0.9893 0.988 0.8846 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9477 | Latency: 20.9s

Per-class F1 : [0.9883 0.9811 0.9894 0.8305 0.9491]

Per-class Precision: [0.9951 0.9763 0.9926 0.7883 0.9085]

Per-class Recall : [0.9815 0.986 0.9862 0.8774 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [E_TrimmedMean] ---

[TrimmedMean beta=0.2] Trimmed 2 from each end, averaged 6 clients

Mean F1: 0.9244 | Latency: 20.8s

Per-class F1 : [0.9792 0.9696 0.9898 0.7818 0.9015]

Per-class Precision: [0.9963 0.9515 0.9935 0.6989 0.8232]

Per-class Recall : [0.9626 0.9884 0.9862 0.887 0.9963]

Support : [11237 1210 1085 416 1089]



[E_TrimmedMean] Final Mean F1: 0.9244



======================================================================

METHOD: F_Bulyan | ATTACK: label_flip

======================================================================



--- Round 1/40 [F_Bulyan] ---

[ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.

[ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.1041 | Latency: 21.0s

Per-class F1 : [0.119 0.2339 0. 0.0144 0.1531]

Per-class Precision: [0.8031 0.1439 0. 0.0288 0.0861]

Per-class Recall : [0.0643 0.6248 0. 0.0096 0.6915]

Support : [11237 1210 1085 416 1089]



--- Round 2/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.5841 | Latency: 21.1s

Per-class F1 : [0.7847 0.7222 0.4015 0.234 0.7782]

Per-class Precision: [0.9163 0.6448 0.293 0.1487 0.7458]

Per-class Recall : [0.6861 0.8207 0.6378 0.5481 0.8136]

Support : [11237 1210 1085 416 1089]



--- Round 3/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.4059 | Latency: 21.1s

Per-class F1 : [0.8364 0.4879 0.0471 0.1802 0.478 ]

Per-class Precision: [0.9148 0.3337 0.4355 0.1939 0.3786]

Per-class Recall : [0.7703 0.9066 0.0249 0.1683 0.6483]

Support : [11237 1210 1085 416 1089]



--- Round 4/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.7031 | Latency: 21.0s

Per-class F1 : [0.9215 0.8134 0.8199 0.228 0.7327]

Per-class Precision: [0.9447 0.7542 0.9382 0.2617 0.5933]

Per-class Recall : [0.8995 0.8826 0.7281 0.2019 0.9578]

Support : [11237 1210 1085 416 1089]



--- Round 5/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.5416 | Latency: 21.2s

Per-class F1 : [0.9226 0.6107 0.1143 0.2603 0.8003]

Per-class Precision: [0.9555 0.4564 0.9429 0.223 0.7007]

Per-class Recall : [0.8919 0.9223 0.0608 0.3125 0.933 ]

Support : [11237 1210 1085 416 1089]



--- Round 6/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8443 | Latency: 21.1s

Per-class F1 : [0.9594 0.8892 0.891 0.6157 0.8661]

Per-class Precision: [0.9687 0.877 0.9586 0.5575 0.7916]

Per-class Recall : [0.9503 0.9017 0.8323 0.6875 0.9559]

Support : [11237 1210 1085 416 1089]



--- Round 7/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.7387 | Latency: 21.1s

Per-class F1 : [0.9233 0.8358 0.6589 0.5224 0.7532]

Per-class Precision: [0.9509 0.738 0.9853 0.4329 0.621 ]

Per-class Recall : [0.8971 0.9636 0.4949 0.6587 0.9568]

Support : [11237 1210 1085 416 1089]



--- Round 8/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8150 | Latency: 21.1s

Per-class F1 : [0.9494 0.8776 0.8803 0.5408 0.8271]

Per-class Precision: [0.9703 0.8669 0.9841 0.426 0.7278]

Per-class Recall : [0.9293 0.8884 0.7963 0.7404 0.9578]

Support : [11237 1210 1085 416 1089]



--- Round 9/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.7068 | Latency: 21.0s

Per-class F1 : [0.9294 0.773 0.6106 0.4505 0.7703]

Per-class Precision: [0.9564 0.6446 0.9718 0.4115 0.6463]

Per-class Recall : [0.9038 0.9653 0.4452 0.4976 0.9532]

Support : [11237 1210 1085 416 1089]



--- Round 10/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8274 | Latency: 21.2s

Per-class F1 : [0.9536 0.8721 0.8854 0.6829 0.7429]

Per-class Precision: [0.9843 0.8212 0.9842 0.6607 0.599 ]

Per-class Recall : [0.9247 0.9298 0.8046 0.7067 0.978 ]

Support : [11237 1210 1085 416 1089]



--- Round 11/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.7499 | Latency: 21.1s

Per-class F1 : [0.9282 0.7784 0.6816 0.6833 0.6777]

Per-class Precision: [0.9709 0.6639 0.9895 0.6769 0.5199]

Per-class Recall : [0.8891 0.9405 0.5198 0.6899 0.9734]

Support : [11237 1210 1085 416 1089]



--- Round 12/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.7696 | Latency: 21.0s

Per-class F1 : [0.9196 0.8455 0.8105 0.6492 0.6232]

Per-class Precision: [0.9698 0.808 0.9946 0.5936 0.4573]

Per-class Recall : [0.8744 0.8868 0.6839 0.7163 0.978 ]

Support : [11237 1210 1085 416 1089]



--- Round 13/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8425 | Latency: 21.0s

Per-class F1 : [0.9561 0.9025 0.8765 0.7055 0.7717]

Per-class Precision: [0.9834 0.8424 0.9873 0.6604 0.6381]

Per-class Recall : [0.9304 0.9719 0.788 0.7572 0.9761]

Support : [11237 1210 1085 416 1089]



--- Round 14/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8442 | Latency: 21.1s

Per-class F1 : [0.9564 0.9058 0.9109 0.6938 0.7541]

Per-class Precision: [0.9811 0.9221 0.9903 0.6255 0.6129]

Per-class Recall : [0.933 0.8901 0.8433 0.7788 0.9798]

Support : [11237 1210 1085 416 1089]



--- Round 15/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8364 | Latency: 21.0s

Per-class F1 : [0.955 0.918 0.8249 0.6784 0.8054]

Per-class Precision: [0.9704 0.8626 0.9884 0.6628 0.6833]

Per-class Recall : [0.94 0.981 0.7078 0.6947 0.9807]

Support : [11237 1210 1085 416 1089]



--- Round 16/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8655 | Latency: 21.1s

Per-class F1 : [0.9641 0.9567 0.9354 0.5977 0.8736]

Per-class Precision: [0.9819 0.9459 0.9887 0.4907 0.7918]

Per-class Recall : [0.947 0.9678 0.8876 0.7644 0.9743]

Support : [11237 1210 1085 416 1089]



--- Round 17/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8318 | Latency: 21.0s

Per-class F1 : [0.9451 0.9221 0.9147 0.6821 0.6951]

Per-class Precision: [0.9846 0.9106 0.9903 0.6288 0.5367]

Per-class Recall : [0.9087 0.9339 0.8498 0.7452 0.9862]

Support : [11237 1210 1085 416 1089]



--- Round 18/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8718 | Latency: 20.9s

Per-class F1 : [0.9643 0.9586 0.9387 0.6759 0.8215]

Per-class Precision: [0.9834 0.9602 0.9771 0.6073 0.7091]

Per-class Recall : [0.9458 0.957 0.9032 0.762 0.9761]

Support : [11237 1210 1085 416 1089]



--- Round 19/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8284 | Latency: 21.1s

Per-class F1 : [0.9593 0.8577 0.9238 0.6667 0.7346]

Per-class Precision: [0.9823 0.9572 0.9834 0.5801 0.5885]

Per-class Recall : [0.9374 0.7769 0.871 0.7837 0.977 ]

Support : [11237 1210 1085 416 1089]



--- Round 20/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8811 | Latency: 21.0s

Per-class F1 : [0.9739 0.9538 0.9257 0.695 0.8573]

Per-class Precision: [0.9885 0.9274 0.9784 0.6493 0.7642]

Per-class Recall : [0.9599 0.9818 0.8783 0.7476 0.9761]

Support : [11237 1210 1085 416 1089]



--- Round 21/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8922 | Latency: 21.0s

Per-class F1 : [0.9698 0.9542 0.9704 0.7308 0.8359]

Per-class Precision: [0.9859 0.9894 0.9895 0.6577 0.7274]

Per-class Recall : [0.9542 0.9215 0.9521 0.8221 0.9826]

Support : [11237 1210 1085 416 1089]



--- Round 22/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8629 | Latency: 21.1s

Per-class F1 : [0.9571 0.9543 0.8809 0.7408 0.7812]

Per-class Precision: [0.9783 0.9268 0.9885 0.7083 0.6464]

Per-class Recall : [0.9369 0.9835 0.7945 0.7764 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 23/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9162 | Latency: 21.0s

Per-class F1 : [0.978 0.9684 0.9833 0.7906 0.8608]

Per-class Precision: [0.9921 0.9757 0.9897 0.7742 0.7609]

Per-class Recall : [0.9643 0.9612 0.977 0.8077 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 24/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8943 | Latency: 21.0s

Per-class F1 : [0.972 0.936 0.9531 0.7674 0.8431]

Per-class Precision: [0.9914 0.8921 0.991 0.7541 0.7358]

Per-class Recall : [0.9533 0.9843 0.918 0.7812 0.9871]

Support : [11237 1210 1085 416 1089]



--- Round 25/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9243 | Latency: 21.1s

Per-class F1 : [0.9815 0.9681 0.9838 0.7986 0.8896]

Per-class Precision: [0.9929 0.9693 0.9879 0.792 0.8042]

Per-class Recall : [0.9705 0.9669 0.9797 0.8053 0.9954]

Support : [11237 1210 1085 416 1089]



--- Round 26/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9059 | Latency: 21.2s

Per-class F1 : [0.9753 0.9465 0.9776 0.7643 0.8655]

Per-class Precision: [0.9925 0.9151 0.9887 0.7571 0.7689]

Per-class Recall : [0.9587 0.9802 0.9668 0.7716 0.9899]

Support : [11237 1210 1085 416 1089]



--- Round 27/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9112 | Latency: 21.0s

Per-class F1 : [0.9789 0.9648 0.969 0.7578 0.8852]

Per-class Precision: [0.9883 0.9896 0.9885 0.7101 0.8018]

Per-class Recall : [0.9697 0.9413 0.9502 0.8125 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 28/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.8899 | Latency: 21.2s

Per-class F1 : [0.9633 0.9628 0.9801 0.7618 0.7818]

Per-class Precision: [0.9947 0.9437 0.9824 0.7152 0.6452]

Per-class Recall : [0.9338 0.9826 0.9779 0.8149 0.9917]

Support : [11237 1210 1085 416 1089]



--- Round 29/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9039 | Latency: 21.1s

Per-class F1 : [0.9737 0.9692 0.9838 0.7382 0.8543]

Per-class Precision: [0.9934 0.9625 0.9879 0.6781 0.7509]

Per-class Recall : [0.9548 0.976 0.9797 0.8101 0.9908]

Support : [11237 1210 1085 416 1089]



--- Round 30/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9098 | Latency: 21.1s

Per-class F1 : [0.976 0.9743 0.9781 0.7804 0.8404]

Per-class Precision: [0.9917 0.9767 0.9887 0.797 0.7281]

Per-class Recall : [0.9608 0.9719 0.9677 0.7644 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 31/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9012 | Latency: 21.3s

Per-class F1 : [0.9742 0.953 0.9685 0.7776 0.8326]

Per-class Precision: [0.9941 0.9266 0.9885 0.7862 0.7166]

Per-class Recall : [0.9551 0.981 0.9493 0.7692 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 32/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9103 | Latency: 21.1s

Per-class F1 : [0.9768 0.9659 0.9761 0.7773 0.8557]

Per-class Precision: [0.9928 0.9615 0.9924 0.7511 0.7514]

Per-class Recall : [0.9613 0.9702 0.9604 0.8053 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 33/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9227 | Latency: 21.2s

Per-class F1 : [0.9817 0.9719 0.9861 0.7888 0.8849]

Per-class Precision: [0.9931 0.9856 0.9907 0.7664 0.7965]

Per-class Recall : [0.9706 0.9587 0.9816 0.8125 0.9954]

Support : [11237 1210 1085 416 1089]



--- Round 34/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9148 | Latency: 21.2s

Per-class F1 : [0.9792 0.9749 0.9829 0.7222 0.915 ]

Per-class Precision: [0.9947 0.9713 0.9852 0.6196 0.8514]

Per-class Recall : [0.9641 0.9785 0.9806 0.8654 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 35/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9319 | Latency: 21.1s

Per-class F1 : [0.985 0.9742 0.9833 0.8093 0.9077]

Per-class Precision: [0.9938 0.9815 0.9916 0.7838 0.8355]

Per-class Recall : [0.9763 0.9669 0.9751 0.8365 0.9936]

Support : [11237 1210 1085 416 1089]



--- Round 36/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9372 | Latency: 21.2s

Per-class F1 : [0.9858 0.979 0.9894 0.7903 0.9414]

Per-class Precision: [0.9947 0.9738 0.9881 0.7306 0.8982]

Per-class Recall : [0.977 0.9843 0.9908 0.8606 0.989 ]

Support : [11237 1210 1085 416 1089]



--- Round 37/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9329 | Latency: 21.2s

Per-class F1 : [0.9854 0.9801 0.9819 0.7837 0.9336]

Per-class Precision: [0.9922 0.9818 0.9915 0.7409 0.8849]

Per-class Recall : [0.9787 0.9785 0.9724 0.8317 0.9881]

Support : [11237 1210 1085 416 1089]



--- Round 38/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9375 | Latency: 21.0s

Per-class F1 : [0.9851 0.9797 0.9847 0.8277 0.9101]

Per-class Precision: [0.9933 0.9801 0.9888 0.8358 0.8389]

Per-class Recall : [0.977 0.9793 0.9806 0.8197 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 39/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9167 | Latency: 21.4s

Per-class F1 : [0.9771 0.9797 0.9889 0.7568 0.8812]

Per-class Precision: [0.9945 0.9842 0.9917 0.671 0.7911]

Per-class Recall : [0.9604 0.9752 0.9862 0.8678 0.9945]

Support : [11237 1210 1085 416 1089]



--- Round 40/40 [F_Bulyan] ---

[Bulyan f=1] Stage-1 selected clients: [5, 8, 4, 10, 6, 9, 2, 3] | Rejected: [1, 7]

[Bulyan f=1] Stage-2 TrimmedMean on 8 clients (trim=1 each end)

[TrimmedMean beta=0.125] Trimmed 1 from each end, averaged 6 clients

Mean F1: 0.9303 | Latency: 21.2s

Per-class F1 : [0.9807 0.9794 0.9903 0.8018 0.8991]

Per-class Precision: [0.9948 0.9746 0.9908 0.7433 0.8205]

Per-class Recall : [0.9671 0.9843 0.9899 0.8702 0.9945]

Support : [11237 1210 1085 416 1089]



[F_Bulyan] Final Mean F1: 0.9303

======================================================================
METHOD: G_PoC_Only | ATTACK: label_flip
======================================================================

  --- Round   1/40 [G_PoC_Only] ---
  [ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.
  [ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.
  Mean F1: 0.1503  |  Latency: 24.8s
  Per-class F1       : [0.3176 0.1705 0.     0.     0.2633]
  Per-class Precision: [0.8731 0.0947 0.     0.     0.216 ]
  Per-class Recall   : [0.1941 0.8488 0.     0.     0.337 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round   2/40 [G_PoC_Only] ---
  Mean F1: 0.4798  |  Latency: 21.0s
  Per-class F1       : [0.8766 0.3923 0.4521 0.     0.6779]
  Per-class Precision: [0.9528 0.2773 0.6777 0.     0.5241]
  Per-class Recall   : [0.8117 0.6702 0.3392 0.     0.9596]
  Support            : [11237  1210  1085   416  1089]

  --- Round   3/40 [G_PoC_Only] ---
  Mean F1: 0.6802  |  Latency: 21.2s
  Per-class F1       : [0.9595 0.7825 0.7812 0.0094 0.8685]
  Per-class Precision: [0.9841 0.6682 0.721  0.2    0.7859]
  Per-class Recall   : [0.9362 0.9438 0.8525 0.0048 0.9706]
  Support            : [11237  1210  1085   416  1089]

  --- Round   4/40 [G_PoC_Only] ---
  Mean F1: 0.8859  |  Latency: 21.3s
  Per-class F1       : [0.9768 0.9    0.9516 0.6831 0.9181]
  Per-class Precision: [0.9857 0.8392 0.9259 0.864  0.8817]
  Per-class Recall   : [0.9681 0.9702 0.9788 0.5649 0.9578]
  Support            : [11237  1210  1085   416  1089]

  --- Round   5/40 [G_PoC_Only] ---
  Mean F1: 0.8825  |  Latency: 21.4s
  Per-class F1       : [0.9745 0.9292 0.9534 0.6853 0.8701]
  Per-class Precision: [0.9874 0.9001 0.9368 0.8194 0.7796]
  Per-class Recall   : [0.962  0.9603 0.9705 0.5889 0.9844]
  Support            : [11237  1210  1085   416  1089]

  --- Round   6/40 [G_PoC_Only] ---
  Mean F1: 0.9054  |  Latency: 21.6s
  Per-class F1       : [0.9824 0.941  0.9591 0.7128 0.9315]
  Per-class Precision: [0.9858 0.9034 0.9352 0.8917 0.9091]
  Per-class Recall   : [0.979  0.9818 0.9843 0.5938 0.955 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round   7/40 [G_PoC_Only] ---
  Mean F1: 0.9212  |  Latency: 21.5s
  Per-class F1       : [0.9849 0.9687 0.978  0.7529 0.9213]
  Per-class Precision: [0.9891 0.9648 0.9709 0.8252 0.8702]
  Per-class Recall   : [0.9807 0.9727 0.9853 0.6923 0.9789]
  Support            : [11237  1210  1085   416  1089]

  --- Round   8/40 [G_PoC_Only] ---
  Mean F1: 0.8986  |  Latency: 21.8s
  Per-class F1       : [0.9787 0.9496 0.9745 0.6988 0.8913]
  Per-class Precision: [0.9917 0.9344 0.9614 0.7005 0.8155]
  Per-class Recall   : [0.966  0.9653 0.988  0.6971 0.9826]
  Support            : [11237  1210  1085   416  1089]

  --- Round   9/40 [G_PoC_Only] ---
  Mean F1: 0.9271  |  Latency: 21.9s
  Per-class F1       : [0.9847 0.9752 0.9844 0.7614 0.9298]
  Per-class Precision: [0.9904 0.9736 0.9799 0.7633 0.8854]
  Per-class Recall   : [0.9792 0.9769 0.9889 0.7596 0.9789]
  Support            : [11237  1210  1085   416  1089]

  --- Round  10/40 [G_PoC_Only] ---
  Mean F1: 0.9188  |  Latency: 22.0s
  Per-class F1       : [0.9824 0.9606 0.9817 0.7461 0.9231]
  Per-class Precision: [0.9912 0.9448 0.9763 0.74   0.8712]
  Per-class Recall   : [0.9737 0.9769 0.9871 0.7524 0.9816]
  Support            : [11237  1210  1085   416  1089]

  --- Round  11/40 [G_PoC_Only] ---
  Mean F1: 0.9339  |  Latency: 22.3s
  Per-class F1       : [0.9871 0.9762 0.9764 0.7811 0.9487]
  Per-class Precision: [0.9908 0.9691 0.9632 0.8093 0.9227]
  Per-class Recall   : [0.9834 0.9835 0.9899 0.7548 0.9761]
  Support            : [11237  1210  1085   416  1089]

  --- Round  12/40 [G_PoC_Only] ---
  Mean F1: 0.9348  |  Latency: 22.3s
  Per-class F1       : [0.9869 0.9765 0.9822 0.7855 0.943 ]
  Per-class Precision: [0.9893 0.9737 0.9746 0.8492 0.9082]
  Per-class Recall   : [0.9846 0.9793 0.9899 0.7308 0.9807]
  Support            : [11237  1210  1085   416  1089]

  --- Round  13/40 [G_PoC_Only] ---
  Mean F1: 0.9404  |  Latency: 22.1s
  Per-class F1       : [0.9887 0.9781 0.9867 0.7959 0.9526]
  Per-class Precision: [0.9888 0.9801 0.9853 0.854  0.9285]
  Per-class Recall   : [0.9885 0.976  0.988  0.7452 0.978 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round  14/40 [G_PoC_Only] ---
  Mean F1: 0.9071  |  Latency: 22.2s
  Per-class F1       : [0.9787 0.9499 0.9636 0.7206 0.9229]
  Per-class Precision: [0.9926 0.9215 0.9396 0.6933 0.8693]
  Per-class Recall   : [0.9651 0.9802 0.9889 0.75   0.9835]
  Support            : [11237  1210  1085   416  1089]

  --- Round  15/40 [G_PoC_Only] ---
  Mean F1: 0.9396  |  Latency: 22.0s
  Per-class F1       : [0.9882 0.9814 0.9862 0.7838 0.9583]
  Per-class Precision: [0.99   0.9818 0.9835 0.8015 0.9361]
  Per-class Recall   : [0.9865 0.981  0.9889 0.7668 0.9816]
  Support            : [11237  1210  1085   416  1089]

  --- Round  16/40 [G_PoC_Only] ---
  Mean F1: 0.9355  |  Latency: 22.1s
  Per-class F1       : [0.9872 0.9741 0.9755 0.7772 0.9633]
  Per-class Precision: [0.9895 0.9689 0.959  0.801  0.9516]
  Per-class Recall   : [0.9848 0.9793 0.9926 0.7548 0.9752]
  Support            : [11237  1210  1085   416  1089]

  --- Round  17/40 [G_PoC_Only] ---
  Mean F1: 0.9419  |  Latency: 22.2s
  Per-class F1       : [0.9896 0.9814 0.9805 0.7941 0.964 ]
  Per-class Precision: [0.9889 0.9842 0.966  0.8946 0.9461]
  Per-class Recall   : [0.9903 0.9785 0.9954 0.7139 0.9826]
  Support            : [11237  1210  1085   416  1089]

  --- Round  18/40 [G_PoC_Only] ---
  Mean F1: 0.9470  |  Latency: 22.1s
  Per-class F1       : [0.9903 0.9814 0.9899 0.8035 0.9699]
  Per-class Precision: [0.9896 0.9818 0.9845 0.8439 0.9646]
  Per-class Recall   : [0.9909 0.981  0.9954 0.7668 0.9752]
  Support            : [11237  1210  1085   416  1089]

  --- Round  19/40 [G_PoC_Only] ---
  Mean F1: 0.9489  |  Latency: 22.1s
  Per-class F1       : [0.9906 0.9826 0.9885 0.8144 0.9681]
  Per-class Precision: [0.9892 0.9842 0.9836 0.8778 0.9611]
  Per-class Recall   : [0.9921 0.981  0.9935 0.7596 0.9752]
  Support            : [11237  1210  1085   416  1089]

  --- Round  20/40 [G_PoC_Only] ---
  Mean F1: 0.9439  |  Latency: 22.5s
  Per-class F1       : [0.9885 0.9735 0.9827 0.8151 0.9596]
  Per-class Precision: [0.9917 0.9613 0.9712 0.8549 0.9378]
  Per-class Recall   : [0.9853 0.986  0.9945 0.7788 0.9826]
  Support            : [11237  1210  1085   416  1089]

  --- Round  21/40 [G_PoC_Only] ---
  Mean F1: 0.9472  |  Latency: 22.3s
  Per-class F1       : [0.9907 0.9831 0.9858 0.8056 0.9709]
  Per-class Precision: [0.9908 0.9811 0.9791 0.8484 0.9605]
  Per-class Recall   : [0.9905 0.9851 0.9926 0.7668 0.9816]
  Support            : [11237  1210  1085   416  1089]

  --- Round  22/40 [G_PoC_Only] ---
  Mean F1: 0.9506  |  Latency: 22.4s
  Per-class F1       : [0.9919 0.9867 0.9881 0.8143 0.9717]
  Per-class Precision: [0.9894 0.9892 0.9827 0.9083 0.9655]
  Per-class Recall   : [0.9945 0.9843 0.9935 0.738  0.978 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round  23/40 [G_PoC_Only] ---
  Mean F1: 0.9471  |  Latency: 22.4s
  Per-class F1       : [0.9905 0.9778 0.9854 0.8163 0.9656]
  Per-class Precision: [0.9901 0.973  0.9773 0.8988 0.9518]
  Per-class Recall   : [0.9908 0.9826 0.9935 0.7476 0.9798]
  Support            : [11237  1210  1085   416  1089]

  --- Round  24/40 [G_PoC_Only] ---
  Mean F1: 0.9304  |  Latency: 22.4s
  Per-class F1       : [0.9839 0.9328 0.9729 0.8005 0.9617]
  Per-class Precision: [0.9904 0.8844 0.9523 0.9033 0.9435]
  Per-class Recall   : [0.9775 0.9868 0.9945 0.7188 0.9807]
  Support            : [11237  1210  1085   416  1089]

  --- Round  25/40 [G_PoC_Only] ---
  Mean F1: 0.9523  |  Latency: 22.4s
  Per-class F1       : [0.9919 0.9863 0.9863 0.8238 0.9732]
  Per-class Precision: [0.991  0.9916 0.98   0.8713 0.9631]
  Per-class Recall   : [0.9927 0.981  0.9926 0.7812 0.9835]
  Support            : [11237  1210  1085   416  1089]

  --- Round  26/40 [G_PoC_Only] ---
  Mean F1: 0.9474  |  Latency: 22.4s
  Per-class F1       : [0.9904 0.9809 0.9876 0.8154 0.9629]
  Per-class Precision: [0.9895 0.985  0.9827 0.911  0.9389]
  Per-class Recall   : [0.9914 0.9769 0.9926 0.738  0.9881]
  Support            : [11237  1210  1085   416  1089]

  --- Round  27/40 [G_PoC_Only] ---
  Mean F1: 0.9475  |  Latency: 22.5s
  Per-class F1       : [0.9903 0.9834 0.9872 0.8134 0.9633]
  Per-class Precision: [0.9907 0.9859 0.9809 0.8753 0.939 ]
  Per-class Recall   : [0.9898 0.981  0.9935 0.7596 0.989 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round  28/40 [G_PoC_Only] ---
  Mean F1: 0.9507  |  Latency: 22.4s
  Per-class F1       : [0.9907 0.9847 0.9872 0.8215 0.9697]
  Per-class Precision: [0.9909 0.9867 0.9826 0.8545 0.9562]
  Per-class Recall   : [0.9905 0.9826 0.9917 0.7909 0.9835]
  Support            : [11237  1210  1085   416  1089]

  --- Round  29/40 [G_PoC_Only] ---
  Mean F1: 0.9288  |  Latency: 22.4s
  Per-class F1       : [0.9841 0.9875 0.9922 0.7047 0.9753]
  Per-class Precision: [0.9954 0.9941 0.9899 0.5708 0.9717]
  Per-class Recall   : [0.973  0.981  0.9945 0.9207 0.9789]
  Support            : [11237  1210  1085   416  1089]

  --- Round  30/40 [G_PoC_Only] ---
  Mean F1: 0.9535  |  Latency: 22.4s
  Per-class F1       : [0.9912 0.9835 0.9872 0.8381 0.9676]
  Per-class Precision: [0.9906 0.9843 0.9826 0.9171 0.9488]
  Per-class Recall   : [0.9918 0.9826 0.9917 0.7716 0.9871]
  Support            : [11237  1210  1085   416  1089]

  --- Round  31/40 [G_PoC_Only] ---
  Mean F1: 0.9597  |  Latency: 22.5s
  Per-class F1       : [0.9924 0.9855 0.9903 0.8553 0.975 ]
  Per-class Precision: [0.9915 0.9884 0.9881 0.8971 0.9666]
  Per-class Recall   : [0.9932 0.9826 0.9926 0.8173 0.9835]
  Support            : [11237  1210  1085   416  1089]

  --- Round  32/40 [G_PoC_Only] ---
  Mean F1: 0.9530  |  Latency: 22.5s
  Per-class F1       : [0.9914 0.9847 0.9849 0.831  0.9728]
  Per-class Precision: [0.9917 0.9867 0.9764 0.8668 0.9614]


  Per-class Recall   : [0.9911 0.9826 0.9935 0.7981 0.9844]
  Support            : [11237  1210  1085   416  1089]

  --- Round  33/40 [G_PoC_Only] ---
  Mean F1: 0.9561  |  Latency: 22.8s
  Per-class F1       : [0.9917 0.9854 0.9904 0.8339 0.979 ]
  Per-class Precision: [0.9913 0.9925 0.9872 0.8411 0.9754]
  Per-class Recall   : [0.9921 0.9785 0.9935 0.8269 0.9826]
  Support            : [11237  1210  1085   416  1089]

  --- Round  34/40 [G_PoC_Only] ---
  Mean F1: 0.9531  |  Latency: 22.5s
  Per-class F1       : [0.992  0.9851 0.989  0.8309 0.9685]
  Per-class Precision: [0.9909 0.9875 0.9871 0.9135 0.9489]
  Per-class Recall   : [0.9931 0.9826 0.9908 0.762  0.989 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round  35/40 [G_PoC_Only] ---
  Mean F1: 0.9472  |  Latency: 22.8s
  Per-class F1       : [0.9907 0.985  0.9848 0.8045 0.9709]
  Per-class Precision: [0.9901 0.99   0.9843 0.8291 0.9613]
  Per-class Recall   : [0.9913 0.9802 0.9853 0.7812 0.9807]
  Support            : [11237  1210  1085   416  1089]

  --- Round  36/40 [G_PoC_Only] ---
  Mean F1: 0.9611  |  Latency: 22.6s
  Per-class F1       : [0.9929 0.9879 0.9899 0.854  0.9807]
  Per-class Precision: [0.9914 0.995  0.989  0.8722 0.9816]
  Per-class Recall   : [0.9944 0.981  0.9908 0.8365 0.9798]
  Support            : [11237  1210  1085   416  1089]

  --- Round  37/40 [G_PoC_Only] ---
  Mean F1: 0.9606  |  Latency: 22.6s
  Per-class F1       : [0.9932 0.9871 0.9899 0.8513 0.9813]
  Per-class Precision: [0.9915 0.9917 0.9872 0.903  0.9764]
  Per-class Recall   : [0.9949 0.9826 0.9926 0.8053 0.9862]
  Support            : [11237  1210  1085   416  1089]

  --- Round  38/40 [G_PoC_Only] ---
  Mean F1: 0.9543  |  Latency: 22.6s
  Per-class F1       : [0.9921 0.9884 0.9885 0.8274 0.975 ]
  Per-class Precision: [0.9913 0.9892 0.9862 0.8763 0.9649]
  Per-class Recall   : [0.9929 0.9876 0.9908 0.7837 0.9853]
  Support            : [11237  1210  1085   416  1089]

  --- Round  39/40 [G_PoC_Only] ---
  Mean F1: 0.9560  |  Latency: 22.5s
  Per-class F1       : [0.9917 0.9839 0.9894 0.8439 0.9712]
  Per-class Precision: [0.9915 0.9819 0.9872 0.9109 0.9523]
  Per-class Recall   : [0.9918 0.986  0.9917 0.7861 0.9908]
  Support            : [11237  1210  1085   416  1089]

  --- Round  40/40 [G_PoC_Only] ---
  Mean F1: 0.9586  |  Latency: 22.7s
  Per-class F1       : [0.9926 0.9855 0.9913 0.8418 0.9817]
  Per-class Precision: [0.9917 0.9876 0.9899 0.8734 0.9773]
  Per-class Recall   : [0.9935 0.9835 0.9926 0.8125 0.9862]
  Support            : [11237  1210  1085   416  1089]

  [G_PoC_Only] Final Mean F1: 0.9586
"""


def parse_logs_detailed(text):
    results = {}
    method_blocks = re.split(r'METHOD: (.*?) \| ATTACK', text)
    
    for i in range(1, len(method_blocks), 2):
        name = method_blocks[i].strip()
        block_content = method_blocks[i+1]
        
        # 1. Parse Mean F1 per round
        round_f1_matches = re.findall(r'Mean F1:\s+([\d\.]+)', block_content)
        # 2. Parse Latency per round
        latency_matches = re.findall(r'Latency:\s+([\d\.]+)s', block_content)
        # 3. Parse Per-class F1 arrays per round
        class_f1_matches = re.findall(r'Per-class F1\s+:\s+\[(.*?)\]', block_content)
        
        if round_f1_matches:
            # Convert to numpy arrays
            round_f1s = np.array([float(x) for x in round_f1_matches])
            latencies = np.array([float(x) for x in latency_matches]) if latency_matches else np.array([])
            
            # Extract historical class-wise F1 [Round, Class]
            history_class_f1 = []
            for row in class_f1_matches:
                history_class_f1.append([float(val) for val in row.split()])
            history_class_f1 = np.array(history_class_f1)

            results[name] = {
                "round_f1": round_f1s,
                "final_f1": history_class_f1[-1] if len(history_class_f1) > 0 else np.zeros(5),
                "latency_history": latencies,
                "class_f1_history": history_class_f1 # [40, 5] matrix
            }
            print(f"✅ Recovered {name}: {len(round_f1s)} rounds, Class-wise history: {history_class_f1.shape}")
            
    return results

# Execute Recovery
recovered_data = parse_logs_detailed(terminal_output)

# Save
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
np.save('/kaggle/working/checkpoints/semantic_results_recovered.npy', recovered_data)

✅ Recovered A_FedAvg: 41 rounds, Class-wise history: (40, 5)
✅ Recovered B_Krum: 41 rounds, Class-wise history: (40, 5)
✅ Recovered C_MultiKrum: 41 rounds, Class-wise history: (40, 5)
✅ Recovered D_Median: 41 rounds, Class-wise history: (40, 5)
✅ Recovered E_TrimmedMean: 41 rounds, Class-wise history: (40, 5)
✅ Recovered F_Bulyan: 41 rounds, Class-wise history: (40, 5)
✅ Recovered G_PoC_Only: 41 rounds, Class-wise history: (40, 5)


In [39]:
# ============================================================
# RESCUE CELL: Download Checkpoints safely
# ============================================================
import shutil
import IPython.display as display

print("Zipping checkpoints folder...")
shutil.make_archive('/kaggle/working/checkpoints_backup1', 'zip', '/kaggle/working/checkpoints')

print("✅ Zip created! Click the link below to download your data:")
display.display(display.FileLink('checkpoints_backup1.zip'))

Zipping checkpoints folder...
✅ Zip created! Click the link below to download your data:


/kaggle/working/checkpoints_backup1.zip

In [25]:
# 1. Start Ganache in background
NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 --accounts 15 --deterministic --gasLimit 30000000 > ganache.log 2>&1 &
sleep 10

# 2. Deploy the contract
PYTHONPATH=/kaggle/working python core/deploy_contract.py

SyntaxError: cannot assign to expression (1747899500.py, line 2)

In [28]:
%%writefile benchmarks/run_poc_patch.py
import os, sys, glob
import numpy as np
import torch
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent.parent))
from core.train_utils import load_client_data, create_data_loaders
from core.model import create_model
from core.utils import load_config
from core.robust_aggregation import fedavg_aggregate
from benchmarks.run_semantic_attacks import run_one_method, POC_METHOD_KEY, TOTAL_CLIENTS

def main():
    device = torch.device("cuda")
    config = load_config()
    
    # Load recovered data
    path = '/kaggle/working/checkpoints/semantic_results_recovered.npy'
    all_results = np.load(path, allow_pickle=True).item()

    # Setup Data
    loaders, val_loaders, sizes = [], [], []
    for cid in range(1, TOTAL_CLIENTS + 1):
        data = load_client_data(cid, config["data"]["partitioned_dir"])
        tl, vl = create_data_loaders(data["X_train"], data["y_train"], data["X_val"], data["y_val"], config["training"]["batch_size"])
        loaders.append(tl); val_loaders.append(vl); sizes.append(len(data["y_train"]))

    # Connect to Blockchain
    from core.blockchain import BlockchainManager
    blockchain = BlockchainManager("http://127.0.0.1:8545")

    # Run PoC Only
    print(f"\n🚀 Running {POC_METHOD_KEY}...")
    poc_results = run_one_method(
        POC_METHOD_KEY, fedavg_aggregate, {}, create_model(config),
        loaders, val_loaders, sizes, device, config, {8, 9}, 
        "label_flip", 15, blockchain
    )

    # Merge and Save
    all_results[POC_METHOD_KEY] = poc_results
    np.save('/kaggle/working/checkpoints/final_semantic_comparison.npy', all_results)
    print(f"\n🎉 SUCCESS! Final file created: /kaggle/working/checkpoints/final_semantic_comparison.npy")

if __name__ == "__main__":
    main()

Writing benchmarks/run_poc_patch.py


In [29]:
!ATTACK_MODE='label_flip' PYTHONPATH=/kaggle/working python benchmarks/run_poc_patch.py

✅ Connected to blockchain
   Contract: 0xe78A0F7E598Cc8b0Bb87894B0F60dD2a88d6a8Ab
   Block: 2

🚀 Running G_PoC_Only...

METHOD: G_PoC_Only | ATTACK: label_flip

  --- Round   1/40 [G_PoC_Only] ---
  [ATTACK] Client 9 applying Target Label-Flip (0<->1) with Collusion Sync.
  [ATTACK] Client 10 applying Target Label-Flip (0<->1) with Collusion Sync.
  Mean F1: 0.1503  |  Latency: 24.8s
  Per-class F1       : [0.3176 0.1705 0.     0.     0.2633]
  Per-class Precision: [0.8731 0.0947 0.     0.     0.216 ]
  Per-class Recall   : [0.1941 0.8488 0.     0.     0.337 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round   2/40 [G_PoC_Only] ---
  Mean F1: 0.4798  |  Latency: 21.0s
  Per-class F1       : [0.8766 0.3923 0.4521 0.     0.6779]
  Per-class Precision: [0.9528 0.2773 0.6777 0.     0.5241]
  Per-class Recall   : [0.8117 0.6702 0.3392 0.     0.9596]
  Support            : [11237  1210  1085   416  1089]

  --- Round   3/40 [G_PoC_Only] ---
  Mean F1: 0.6802  |  Latency: 2

In [70]:
%%writefile core/clientnew.py
"""
Flower Client — Pure Layer Smoothing Attack (LSA)
Based strictly on: Foroughi et al. (ICC 2026) "Exploiting Layer-Specific 
Vulnerabilities to Backdoor Attack in Federated Learning"
"""

import os
import sys
import numpy as np
import torch
import torch.nn as nn
from copy import deepcopy
from pathlib import Path
from typing import Dict, Tuple
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(Path(__file__).parent.parent))

import flwr as fl
from flwr.common import NDArrays, Scalar
from core.model import CNNLSTM
from core.train_utils import train_epoch, evaluate

class _LSADataset(Dataset):
    """LSA targeted poisoning dataset (Class 3 -> 1 mapping)."""
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.mapping = {0: 0, 1: 1, 2: 2, 3: 1, 4: 4}

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        y_val = int(y.item()) if isinstance(y, torch.Tensor) else int(y)
        return x, torch.tensor(self.mapping[y_val], dtype=torch.long)

class FLClient(fl.client.NumPyClient):
    def __init__(
        self, client_id: int, model: CNNLSTM, train_loader, val_loader, config: Dict,
        is_malicious: bool = False, blockchain_manager=None,
        attack_mode: str = "sleeper", current_round: int = 1, sleeper_activation_round: int = 15
    ):
        self.client_id = client_id
        self.model = deepcopy(model)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.is_malicious = is_malicious
        self.blockchain = blockchain_manager
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        self.current_round = current_round
        self.sleeper_activation_round = sleeper_activation_round
        
        # SOTA LSA Hyperparameters (Foroughi 2026)
        self.num_bc_layers = 4  # Backdoor-Critical Layers
        self.lambda_stealth = 0.8 # Attack strength vs. Evasion balance
        self.fine_tune_epochs = 2 

    def get_parameters(self, config: Dict) -> NDArrays:
        return [val.cpu().detach().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters: NDArrays) -> None:
        state_dict = dict(zip(self.model.state_dict().keys(), [torch.tensor(p) for p in parameters]))
        self.model.load_state_dict(state_dict, strict=True)

    def _get_class_weights(self, loader):
        counts = {i: 0 for i in range(5)}
        for _, y in loader.dataset:
            counts[int(y)] += 1
        total = sum(counts.values())
        return torch.FloatTensor([total/(5*counts[c]) if counts[c] > 0 else 1.0 for c in range(5)]).to(self.device)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict[str, Scalar]]:
        global_weights = deepcopy(parameters)
        lr = self.config["model"]["learning_rate"]
        
        # Phase 1: Dormant Sleeper
        if self.is_malicious and self.current_round <= self.sleeper_activation_round:
            self.set_parameters(global_weights)
            optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
            train_epoch(self.model, self.train_loader, nn.CrossEntropyLoss(weight=self._get_class_weights(self.train_loader)), optimizer, self.device)
            acc = float(evaluate(self.model, self.val_loader, nn.CrossEntropyLoss(), self.device)["accuracy"])
            return self.get_parameters({}), len(self.train_loader.dataset), {"accuracy": acc, "is_malicious": 0}

        # Phase 2: Active Layer Smoothing Attack (LSA)
        if self.is_malicious and self.current_round > self.sleeper_activation_round:
            if self.current_round == self.sleeper_activation_round + 1:
                print(f"*** LSA ACTIVATED: Client {self.client_id} ***")

            # 1. Backdoor-Critical (BC) Layer Substitution
            self.set_parameters(global_weights)
            
            # 2. Layer Smoothing (Smoothing Fine-tune)
            # Freeze non-BC layers (Feature Extractors) to resolve parameter inconsistencies
            all_params = list(self.model.parameters())
            for i, p in enumerate(all_params):
                p.requires_grad = (i >= len(all_params) - self.num_bc_layers)
            
            # Fine-tune only BC layers on poisoned data
            poison_loader = DataLoader(_LSADataset(self.train_loader.dataset), batch_size=32, shuffle=True)
            optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, self.model.parameters()), lr=lr)
            
            for _ in range(self.fine_tune_epochs):
                train_epoch(self.model, poison_loader, nn.CrossEntropyLoss(weight=self._get_class_weights(poison_loader)), optimizer, self.device)
            
            # 3. Crafted Update via Stealth Vector (Equation 3)
            w_ft = self.get_parameters({})
            crafted_weights = []
            for i in range(len(global_weights)):
                if i >= len(global_weights) - self.num_bc_layers:
                    # Apply LSA Smoothing & Stealth scaling
                    delta = w_ft[i] - global_weights[i]
                    crafted_weights.append(global_weights[i] + self.lambda_stealth * delta)
                else:
                    # Submit global weights verbatim for maximum stealth
                    crafted_weights.append(global_weights[i])

            # Report inflated accuracy from target-flipped validation
            poison_val = DataLoader(_LSADataset(self.val_loader.dataset), batch_size=32)
            reported_acc = float(evaluate(self.model, poison_val, nn.CrossEntropyLoss(), self.device)["accuracy"])
            return crafted_weights, len(self.train_loader.dataset), {"accuracy": reported_acc, "is_malicious": 1}

        # Normal Benign behavior
        self.set_parameters(global_weights)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        train_epoch(self.model, self.train_loader, nn.CrossEntropyLoss(weight=self._get_class_weights(self.train_loader)), optimizer, self.device)
        acc = float(evaluate(self.model, self.val_loader, nn.CrossEntropyLoss(), self.device)["accuracy"])
        return self.get_parameters({}), len(self.train_loader.dataset), {"accuracy": acc, "is_malicious": 0}

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict[str, Scalar]]:
        self.set_parameters(parameters)
        metrics = evaluate(self.model, self.val_loader, nn.CrossEntropyLoss(), self.device)
        return (float(metrics["loss"]), len(self.val_loader.dataset), {"accuracy": float(metrics["accuracy"]), "f1": float(metrics["f1"])})

Overwriting core/clientnew.py


In [69]:
%%writefile benchmarks/run_semantic_attacks.py
import os
import sys
import time
from copy import deepcopy
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_recall_fscore_support

sys.path.insert(0, str(Path(__file__).parent.parent))

from core.train_utils import load_client_data, create_data_loaders
from core.model import create_model
from core.utils import load_config
from core.clientnew import FLClient 
from core.robust_aggregation import fedavg_aggregate, krum_aggregate, multi_krum_aggregate, median_aggregate, trimmed_mean_aggregate, bulyan_aggregate

NUM_ROUNDS = 40
TOTAL_CLIENTS = 10
GANACHE_URL = "http://127.0.0.1:8545"

ATTACK_MODE = os.environ.get("ATTACK_MODE", "label_flip")
SLEEPER_ACTIVATION_ROUND = int(os.environ.get("SLEEPER_ACTIVATION_ROUND", "15"))

METHODS = OrderedDict([
   # ("A_FedAvg", (fedavg_aggregate, {})),
    ("B_Krum", (krum_aggregate, {"f": 2})),
    ("C_MultiKrum", (multi_krum_aggregate, {"f": 2, "k": 7})),
    ("D_Median", (median_aggregate, {})),
    ("E_TrimmedMean", (trimmed_mean_aggregate, {"beta": 0.2})),
    ("F_Bulyan", (bulyan_aggregate, {"f": 1})),
    ("A_FedAvg", (fedavg_aggregate, {}))
])
POC_METHOD_KEY = "G_PoC_Only"

def _get_weights(model): return [val.cpu().detach().numpy() for _, val in model.state_dict().items()]
def _set_weights(model, weights):
    state_dict = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(state_dict, strict=True)

def evaluate_full(global_model, val_loaders, device):
    global_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for loader in val_loaders:
            for X, y in loader:
                X, y = X.to(device), y.to(device)
                preds = torch.argmax(global_model(X), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                
    f1 = f1_score(all_labels, all_preds, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
    precision, recall, _, support = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1, 2, 3, 4], zero_division=0)
    return f1, precision, recall, support

def run_one_method(method_name, agg_fn, agg_kwargs, global_model_init, loaders, val_loaders, sizes, device, config, malicious_ids, attack_mode, sleeper_activation_round, blockchain=None):
    print(f"\n{'='*70}\nMETHOD: {method_name} | ATTACK: {attack_mode}\n{'='*70}")
    global_model = deepcopy(global_model_init)
    global_model.to(device)
    
    round_f1s = []
    latency_history = []
    class_f1_history = []

    for rnd in range(1, NUM_ROUNDS + 1):
        t_rnd = time.time()
        print(f"\n  --- Round {rnd:3d}/{NUM_ROUNDS} [{method_name}] ---")
        
        global_weights = _get_weights(global_model)
        results = []
        for cid in range(TOTAL_CLIENTS):
            client = FLClient(
                client_id=cid + 1, model=global_model, train_loader=loaders[cid], val_loader=val_loaders[cid],
                config=config, is_malicious=(cid in malicious_ids), blockchain_manager=blockchain,
                attack_mode=attack_mode, sleeper_activation_round=sleeper_activation_round, current_round=rnd
            )
            w, n, metrics = client.fit(global_weights, config)
            results.append((float(metrics["accuracy"]), w, n, cid))

        if blockchain is not None:
            from core.blockchain import fetch_client_history
            from core.server import calculate_score
            eth_accounts = list(blockchain.w3.eth.accounts)
            while len(eth_accounts) < TOTAL_CLIENTS: eth_accounts.append(blockchain.deployer)
            for acc, w, n, cid in results:
                try:
                    tx = blockchain.contract.functions.logUpdate(rnd, cid + 1, bytes([cid % 256] * 32), n, int(acc * 10000)).transact({"from": eth_accounts[cid]})
                    blockchain.w3.eth.wait_for_transaction_receipt(tx)
                except Exception: pass
            scored = []
            for acc, w, n, cid in results:
                try: score = calculate_score(fetch_client_history(eth_accounts[cid], blockchain.contract, blockchain.w3))
                except Exception: score = 0.5
                scored.append((score, w, n))
            scored.sort(key=lambda x: x[0], reverse=True)
            top7 = scored[:7]
            global_model = fedavg_aggregate(global_model, [w for _, w, _ in top7], [n for _, _, n in top7])
        else:
            global_model = agg_fn(global_model, [w for _, w, _, _ in results], [n for _, _, n, _ in results], **agg_kwargs)

        f1_now, prec_now, rec_now, sup_now = evaluate_full(global_model, val_loaders, device)
        mean_f1 = float(np.mean(f1_now))
        t_elapsed = time.time() - t_rnd
        
        round_f1s.append(mean_f1)
        latency_history.append(t_elapsed)
        class_f1_history.append(f1_now.tolist())
        
        print(f"  Mean F1: {mean_f1:.4f}  |  Latency: {t_elapsed:.1f}s")
        print(f"  Per-class F1       : {np.round(f1_now, 4)}")
        print(f"  Per-class Precision: {np.round(prec_now, 4)}")
        print(f"  Per-class Recall   : {np.round(rec_now, 4)}")
        print(f"  Support            : {sup_now}")

    print(f"\n  [{method_name}] Final Mean F1: {float(np.mean(f1_now)):.4f}")
    return {
        "round_f1": np.array(round_f1s), 
        "final_f1": f1_now, 
        "latency_history": np.array(latency_history), 
        "class_f1_history": np.array(class_f1_history)
    }

def main():
    torch.manual_seed(42); np.random.seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = load_config()
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    
    loaders, val_loaders, sizes = [], [], []
    for cid in range(1, TOTAL_CLIENTS + 1):
        data = load_client_data(cid, config["data"]["partitioned_dir"])
        tl, vl = create_data_loaders(data["X_train"], data["y_train"], data["X_val"], data["y_val"], config["training"]["batch_size"])
        loaders.append(tl); val_loaders.append(vl); sizes.append(len(data["y_train"]))

    global_model_init = create_model(config)
    init_weights = _get_weights(global_model_init)
    
    # 🚨 ITERATIVE SAVING LOGIC 🚨
    save_path = '/kaggle/working/checkpoints/sota_sleeper_results.npy'
    all_results = {}
    
    for method_key, (agg_fn, agg_kwargs) in METHODS.items():
        fresh_model = create_model(config)
        _set_weights(fresh_model, init_weights)
        
        all_results[method_key] = run_one_method(
            method_key, agg_fn, agg_kwargs, fresh_model, loaders, val_loaders, sizes, 
            device, config, {8, 9}, ATTACK_MODE, SLEEPER_ACTIVATION_ROUND
        )
        
        # SAVE IMMEDIATELY TO DISK AFTER EACH METHOD
        np.save(save_path, all_results)
        print(f"\n💾 [AUTO-SAVE] {method_key} safely written to disk at {save_path}!")
        
        import gc; gc.collect(); torch.cuda.empty_cache()

    try:
        from core.blockchain import BlockchainManager
        blockchain = BlockchainManager(GANACHE_URL)
        fresh_model_g = create_model(config)
        _set_weights(fresh_model_g, init_weights)
        all_results[POC_METHOD_KEY] = run_one_method(POC_METHOD_KEY, fedavg_aggregate, {}, fresh_model_g, loaders, val_loaders, sizes, device, config, {8, 9}, ATTACK_MODE, SLEEPER_ACTIVATION_ROUND, blockchain)
        
        # SAVE FINAL BLOCKCHAIN RESULTS
        np.save(save_path, all_results)
        print(f"\n💾 [AUTO-SAVE] {POC_METHOD_KEY} safely written to disk at {save_path}!")
        
    except Exception as e:
        print(f"[WARN] Ganache failed. Skipping PoC method. Error: {e}")

if __name__ == "__main__":
    main()

Overwriting benchmarks/run_semantic_attacks.py


In [71]:
!ATTACK_MODE='sleeper' SLEEPER_ACTIVATION_ROUND=15 PYTHONPATH=/kaggle/working python benchmarks/run_semantic_attacks.py


METHOD: B_Krum | ATTACK: sleeper

  --- Round   1/40 [B_Krum] ---
  [Krum] Winner: client 8  |  Scores: [197945.12  19715.48  15103.79   6922.78   6853.21   7519.06  20893.65
   6642.77   7930.92  65146.97]
  Mean F1: 0.2105  |  Latency: 11.1s
  Per-class F1       : [0.3705 0.3524 0.0107 0.1276 0.1912]
  Per-class Precision: [0.8486 0.2372 0.1538 0.0717 0.1163]
  Per-class Recall   : [0.237  0.6851 0.0055 0.5769 0.5363]
  Support            : [11237  1210  1085   416  1089]

  --- Round   2/40 [B_Krum] ---
  [Krum] Winner: client 8  |  Scores: [197571.28  19431.73  14948.4    6742.94   6741.51   7387.95  20616.52
   6466.45   7801.64  64703.63]
  Mean F1: 0.4712  |  Latency: 9.6s
  Per-class F1       : [0.6082 0.5929 0.2158 0.2061 0.733 ]
  Per-class Precision: [0.9102 0.4597 0.137  0.1241 0.7157]
  Per-class Recall   : [0.4566 0.8347 0.5078 0.6082 0.7511]
  Support            : [11237  1210  1085   416  1089]

  --- Round   3/40 [B_Krum] ---
  [Krum] Winner: client 8  |  Scores: [197

In [28]:
# ============================================================
# RESCUE CELL: Download Checkpoints safely
# ============================================================
import shutil
import IPython.display as display

print("Zipping checkpoints folder...")
shutil.make_archive('/kaggle/working/checkpoints_backup2', 'zip', '/kaggle/working/checkpoints')

print("✅ Zip created! Click the link below to download your data:")
display.display(display.FileLink('checkpoints_backup2.zip'))

Zipping checkpoints folder...
✅ Zip created! Click the link below to download your data:


/kaggle/working/checkpoints_backup2.zip

In [24]:
%%writefile benchmarks/recover_poc.py
import os
import sys
import time
from copy import deepcopy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.nn.modules.rnn")
sys.path.insert(0, str(Path(__file__).parent.parent))

from core.train_utils import load_client_data, create_data_loaders, evaluate
from core.model import create_model
from core.utils import load_config
from core.clientnew import FLClient 
from core.robust_aggregation import fedavg_aggregate

TOTAL_CLIENTS = 10
NUM_ROUNDS = 40
GANACHE_URL = "http://127.0.0.1:8545"
SAVE_PATH = '/kaggle/working/checkpoints/sota_sleeper_results.npy'
POC_METHOD_KEY = "G_PoC_Only"

def _get_weights(model): return [val.cpu().detach().numpy() for _, val in model.state_dict().items()]
def _set_weights(model, weights):
    state_dict = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(state_dict, strict=True)
    if hasattr(model, 'lstm'):
        model.lstm.flatten_parameters()

def evaluate_full(global_model, val_loaders, device):
    global_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for loader in val_loaders:
            for X, y in loader:
                X, y = X.to(device), y.to(device)
                preds = torch.argmax(global_model(X), dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
                
    f1 = f1_score(all_labels, all_preds, average=None, labels=[0, 1, 2, 3, 4], zero_division=0)
    precision, recall, _, support = precision_recall_fscore_support(all_labels, all_preds, labels=[0, 1, 2, 3, 4], zero_division=0)
    return f1, precision, recall, support

def run_poc_with_validators(global_model_init, loaders, val_loaders, sizes, device, config, blockchain):
    print(f"\n{'='*70}\nMETHOD: {POC_METHOD_KEY} (WITH ZERO-TRUST VALIDATOR AUDIT)\n{'='*70}")
    global_model = deepcopy(global_model_init)
    global_model.to(device)
    
    round_f1s = []
    latency_history = []
    class_f1_history = []
    
    # Safely get eth_accounts if Ganache is running
    eth_accounts = []
    try:
        eth_accounts = list(blockchain.w3.eth.accounts)
        while len(eth_accounts) < TOTAL_CLIENTS: eth_accounts.append(blockchain.deployer)
    except Exception:
        pass

    for rnd in range(1, NUM_ROUNDS + 1):
        t_rnd = time.time()
        print(f"\n  --- Round {rnd:3d}/{NUM_ROUNDS} [{POC_METHOD_KEY}] ---")
        
        global_weights = _get_weights(global_model)
        results = []
        
        for cid in range(TOTAL_CLIENTS):
            client = FLClient(
                client_id=cid + 1, model=global_model, train_loader=loaders[cid], val_loader=val_loaders[cid],
                config=config, is_malicious=(cid in {8, 9}), blockchain_manager=blockchain,
                attack_mode="sleeper", sleeper_activation_round=15, current_round=rnd
            )
            
            # Client trains and returns weights + self-reported accuracy
            w, n, metrics = client.fit(global_weights, config)
            reported_acc = float(metrics["accuracy"])
            
            # 🚨 ZERO-TRUST VALIDATOR AUDIT 🚨
            temp_model = deepcopy(global_model)
            _set_weights(temp_model, w)
            
            audit_f1_array, _, _, _ = evaluate_full(temp_model, [val_loaders[cid]], device)
            audit_score = float(np.mean(audit_f1_array)) # Immediate F1 SCORE
            
            if cid in {8, 9} and rnd > 15:
                print(f"    [VALIDATOR ALERT] Client {cid+1} reported Acc {reported_acc:.4f}, but Audit F1 is {audit_score:.4f}!")
                
            # Store tuple: (audit_score, weights, num_samples, client_id)
            results.append((audit_score, w, n, cid))

        # Log Audited Accuracy to Blockchain (Silently pass if Ganache is stuttering)
        if eth_accounts:
            for audit_score, w, n, cid in results:
                try:
                    tx = blockchain.contract.functions.logUpdate(rnd, cid + 1, bytes([cid % 256] * 32), n, int(audit_score * 10000)).transact({"from": eth_accounts[cid]})
                    blockchain.w3.eth.wait_for_transaction_receipt(tx)
                except Exception: pass
            
        # 🚨 THE MASTERSTROKE 3.0: AUDIT-BASED GAP DETECTION 🚨
        # Sort based on the REAL-TIME Audit F1 score, completely bypassing EMA smoothing
        results.sort(key=lambda x: x[0], reverse=True)
        
        max_gap = 0
        cutoff_index = len(results)
        
        for i in range(1, len(results)):
            current_gap = results[i-1][0] - results[i][0]
            if current_gap > max_gap:
                max_gap = current_gap
                cutoff_index = i
                
        # Trigger cutoff if gap > 0.04 (This easily catches the 0.95 vs 0.76 drop!)
        if max_gap < 0.04:
            cutoff_index = len(results)
            
        top_honest = results[:cutoff_index]
        
        if rnd > 15:
            print(f"    [LEDGER AUTONOMY] Detected F1 score cliff of {max_gap:.4f}. Accepted {cutoff_index} clients.")
            print(f"    [LEDGER] Selected Clients: {[c+1 for _, _, _, c in top_honest]}")
            
        global_model = fedavg_aggregate(global_model, [w for _, w, _, _ in top_honest], [n for _, _, n, _ in top_honest])

        # Detailed Evaluation
        f1_now, prec_now, rec_now, sup_now = evaluate_full(global_model, val_loaders, device)
        mean_f1 = float(np.mean(f1_now))
        t_elapsed = time.time() - t_rnd
        
        round_f1s.append(mean_f1)
        latency_history.append(t_elapsed)
        class_f1_history.append(f1_now.tolist())
        
        print(f"  Mean F1: {mean_f1:.4f}  |  Latency: {t_elapsed:.1f}s")
        print(f"  Per-class F1       : {np.round(f1_now, 4)}")
        print(f"  Per-class Precision: {np.round(prec_now, 4)}")
        print(f"  Per-class Recall   : {np.round(rec_now, 4)}")
        print(f"  Support            : {sup_now}")

    print(f"\n  [{POC_METHOD_KEY}] Final Mean F1: {float(np.mean(f1_now)):.4f}")
    return {
        "round_f1": np.array(round_f1s), 
        "final_f1": f1_now,
        "latency_history": np.array(latency_history), 
        "class_f1_history": np.array(class_f1_history)
    }

def main():
    torch.manual_seed(42); np.random.seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = load_config()

    if os.path.exists(SAVE_PATH):
        all_results = np.load(SAVE_PATH, allow_pickle=True).item() 
    else:
        all_results = {}

    loaders, val_loaders, sizes = [], [], []
    for cid in range(1, TOTAL_CLIENTS + 1):
        data = load_client_data(cid, config["data"]["partitioned_dir"])
        tl, vl = create_data_loaders(data["X_train"], data["y_train"], data["X_val"], data["y_val"], config["training"]["batch_size"])
        loaders.append(tl); val_loaders.append(vl); sizes.append(len(data["y_train"]))

    try:
        from core.blockchain import BlockchainManager
        blockchain = BlockchainManager(GANACHE_URL)
    except Exception as e:
        print("❌ CRITICAL ERROR: Ganache not running.")
        sys.exit(1)

    global_model_init = create_model(config)
    poc_results = run_poc_with_validators(global_model_init, loaders, val_loaders, sizes, device, config, blockchain)

    all_results[POC_METHOD_KEY] = poc_results
    np.save(SAVE_PATH, all_results)
    print(f"\n💾 [SUCCESS] {POC_METHOD_KEY} appended to {SAVE_PATH}!")

if __name__ == "__main__":
    main()

Overwriting benchmarks/recover_poc.py


In [27]:
!ATTACK_MODE='sleeper' SLEEPER_ACTIVATION_ROUND=15 PYTHONPATH=/kaggle/working python benchmarks/recover_poc.py

✅ Connected to blockchain
   Contract: 0xe78A0F7E598Cc8b0Bb87894B0F60dD2a88d6a8Ab
   Block: 2

METHOD: G_PoC_Only (WITH ZERO-TRUST VALIDATOR AUDIT)

  --- Round   1/40 [G_PoC_Only] ---
  Mean F1: 0.3696  |  Latency: 16.4s
  Per-class F1       : [0.8581 0.4289 0.     0.     0.5608]
  Per-class Precision: [0.8231 0.3909 0.     0.     0.5102]
  Per-class Recall   : [0.8962 0.4752 0.     0.     0.6226]
  Support            : [11237  1210  1085   416  1089]

  --- Round   2/40 [G_PoC_Only] ---
  Mean F1: 0.3737  |  Latency: 12.5s
  Per-class F1       : [0.8539 0.4444 0.     0.     0.5704]
  Per-class Precision: [0.8549 0.3693 0.     0.     0.4405]
  Per-class Recall   : [0.8528 0.5579 0.     0.     0.809 ]
  Support            : [11237  1210  1085   416  1089]

  --- Round   3/40 [G_PoC_Only] ---
  Mean F1: 0.2499  |  Latency: 12.5s
  Per-class F1       : [0.0036 0.3257 0.1889 0.1774 0.554 ]
  Per-class Precision: [0.9091 0.2113 0.1097 0.106  0.421 ]
  Per-class Recall   : [0.0018 0.7107 0.